In [1]:
#!/usr/bin/env python3
"""Local Linux smoke runs using the repository's gcp_setup_stellar_private.sh.

Edit the settings below, then run: python3 run_local_protocols.py
Requires the repository's build dependencies. No GCP, commit or push operations.
All replicas share one host, so these are functional tests, not distributed benchmarks.
"""

import csv
import fcntl
import hashlib
import json
import os
from pathlib import Path
import re
import shlex
import signal
import socket
import subprocess
import tempfile
import time
from datetime import datetime
from urllib.request import build_opener, ProxyHandler


# ---------------- Settings: edit these ----------------
REPO = Path.home() / "stellar-core"
RESULTS_ROOT = Path.home() / "stellar-local-results"
NUM_NODES = 4                 # Start with 4; use 8 to match your GCP script.
PROTOCOLS = [ "pbft_fixed", "pbft_rotating"]
# Add "simple_it_bracha" to exercise Simple-IT's non-optimistic variant.
BUILD_JOBS = 8
REGENERATE_BUILD = False       # Needed on first run after the source-file split.
CONFIGURE_ARGS = None         # None reuses ./config.status --config, if present.
# For a fresh checkout, set your normal options, e.g. CONFIGURE_ARGS = [].
RESTORE_ORIGINAL_BUILD = False  # Restore header AND rebuild original mode afterward.
CXX = os.environ.get("CXX", "g++")

WORKLOAD = "WORKLOAD_W"       # Write-only exercises consensus for every request.
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 60
CLIENT_DRAIN_TIMEOUT_SEC = 15
STARTUP_TIMEOUT_SEC = 120
SEND_INTERVAL_US = 0
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 1, "max_in_flight": 150},
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]
SCP_OBSERVATION_SEC = 60       # Native diagnostic; no shab_client for SCP.
# ------------------------------------------------------

MODE_FLAGS = {
    "pbft_fixed": "PBFT_MODE", "pbft_rotating": "PBFT_MODE",
    "iths": "ITHS_MODE", "pbft": "PBFT_MODE", "scp": "ENABLE_SCP_TRACKING",
    "tetrabft": "TETRABFT_MODE", "simple_it": "SIMPLE_IT_MODE",
    "forget_it": "FORGET_IT_MODE", "simple_it_bracha": "SIMPLE_IT_MODE",
}
COMMIT_TAGS = {
    "shabdiz": ("[FAST COMMITTED]",),
    # Accept both the legacy and native blog adapter's commit labels.
    "iths": ("[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"),
    "pbft_fixed": ("[PBFT COMMITTED-LOCAL]",), "pbft_rotating": ("[PBFT COMMITTED-LOCAL]",),
    "pbft": ("[PBFT COMMITTED-LOCAL]",), "tetrabft": ("[TETRA COMMITTED]",),
    "simple_it": ("[SIMPLE-IT COMMITTED]",), "simple_it_bracha": ("[SIMPLE-IT COMMITTED]",),
    "forget_it": ("[FORGET-IT COMMITTED]",),
}
HTTP = build_opener(ProxyHandler({}))  # Local endpoints must bypass HTTP proxies.


def save_json(path, value):
    path.write_text(json.dumps(value, indent=2) + "\n")


def replace_once(text, pattern, replacement):
    result, count = re.subn(pattern, lambda _: replacement, text, flags=re.MULTILINE)
    if count != 1:
        raise RuntimeError(f"Expected one match for {pattern!r}, found {count}; check repository changes")
    return result


def set_constant(text, name, value):
    prefix = re.search(rf"^inline constexpr [^\n=]+\b{re.escape(name)}\s*=", text,
                       re.MULTILINE)
    if prefix is None:
        raise RuntimeError(f"Missing experiment setting: {name}")
    return replace_once(text, rf"^inline constexpr [^\n=]+\b{re.escape(name)}\s*=\s*[^;]+;",
                        prefix.group(0) + " " + str(value) + ";")


def select_protocol(original, protocol):
    text = original
    for flag in set(MODE_FLAGS.values()):
        text = set_constant(text, flag, "true" if MODE_FLAGS.get(protocol) == flag else "false")
    for name, value in {
        "PBFT_ROTATING": "true" if protocol == "pbft_rotating" else "false",
        "SIMPLE_IT_OPTIMISTIC": "false" if protocol == "simple_it_bracha" else "true",
        "NonPrune_MODE": "false", "currentWorkload": WORKLOAD,
        "SERVER_BATCH_SIZE": SERVER_BATCH_SIZE, "CLIENT_PORT": CLIENT_PORT,
    }.items():
        text = set_constant(text, name, value)
    return text


def stop_processes(processes):
    # Only process groups started by THIS script; never pkill other experiments.
    for proc in processes:
        if proc.poll() is None:
            try:
                os.killpg(proc.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
    deadline = time.monotonic() + 5
    for proc in processes:
        try:
            proc.wait(timeout=max(0.01, deadline - time.monotonic()))
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait()


def command(args, cwd, log):
    with log.open("a") as out:
        out.write("\n$ " + shlex.join(map(str, args)) + "\n")
        out.flush()
        proc = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = proc.wait()
            if code:
                raise RuntimeError(f"Command failed ({code}); see {log}")
        finally:
            stop_processes([proc])


def launch(args, cwd, log):
    with log.open("w") as out:
        return subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)


def ensure_alive(processes):
    for i, proc in enumerate(processes, 1):
        if proc.poll() is not None:
            raise RuntimeError(f"Replica {i} exited with code {proc.returncode}; inspect its logs")


def check_ports(ports):
    if len(set(ports)) != len(ports):
        raise RuntimeError("Peer, HTTP, and client ports overlap")
    for port in ports:
        with socket.socket() as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("0.0.0.0", port))
            except OSError as exc:
                raise RuntimeError(f"Port {port} is occupied; stop the previous local run") from exc


def local_setup_text(source, root):
    # Reuse your generator, including its seeds, validator lists and port arithmetic.
    text = replace_once(source, r"^STELLAR_CORE=.*$", "STELLAR_CORE=" + shlex.quote(str(REPO / "src/stellar-core")))
    text = replace_once(text, r"^BASE_DIR=.*$", "BASE_DIR=" + shlex.quote(str(root / "nodes")))
    text = replace_once(text, r"^mapfile -t NODE_IPS <.*$",
                        "mapfile -t NODE_IPS < " + shlex.quote(str(root / "localhost_ips.txt")))
    return text.replace('$STELLAR_CORE gen-seed', '"$STELLAR_CORE" gen-seed')


def prepare_nodes(root, setup_source):
    root.mkdir(parents=True)
    (root / "localhost_ips.txt").write_text("127.0.0.1\n" * NUM_NODES)
    setup = root / "setup-local.sh"
    setup.write_text(local_setup_text(setup_source, root))
    command(["bash", setup, "start"], root, root / "setup.log")
    configs, ports = [], []
    for i in range(1, NUM_NODES + 1):
        cfg = root / "nodes" / f"node{i}" / "stellar-core.cfg"
        text = cfg.read_text()
        # All added settings must precede the generator's TOML table headers.
        additions = {
            "SEND_CUSTOM_MESSAGE": "true" if i == 1 else "false",
            "ALLOW_LOCALHOST_FOR_TESTING": "true", "PUBLIC_HTTP_PORT": "false",
            "TARGET_PEER_CONNECTIONS": str(NUM_NODES - 1),
            "MAX_ADDITIONAL_PEER_CONNECTIONS": str(8 * NUM_NODES),
        }
        for key, value in additions.items():
            if re.search(rf"^{key}\s*=", text, re.MULTILINE):
                text = replace_once(text, rf"^{key}\s*=.*$", f"{key}={value}")
            else:
                text = f"{key}={value}\n" + text
        cfg.write_text(text)
        peer = int(re.search(r"^PEER_PORT=(\d+)", text, re.MULTILINE)[1])
        http = int(re.search(r"^HTTP_PORT=(\d+)", text, re.MULTILINE)[1])
        ports.extend([peer, http])
        configs.append((cfg, http))
    check_ports(ports + [CLIENT_PORT])
    # The generator's init branch continues on new-db failure. Check each command here.
    for cfg, _ in configs:
        command([REPO / "src/stellar-core", "new-db", "--conf", cfg], cfg.parent,
                cfg.parent / "init.log")
    return configs


def wait_for_network(replicas, configs, custom):
    deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
    while time.monotonic() < deadline:
        ensure_alive(replicas)
        logs = [(cfg.parent / "stellar-core.log").read_text(errors="replace")
                if (cfg.parent / "stellar-core.log").exists() else "" for cfg, _ in configs]
        if all("Shabdiz started" in log for log in logs):
            if not custom or f"Client listener started on port {CLIENT_PORT}" in logs[0]:
                return
        time.sleep(1)
    raise TimeoutError("Full peer connectivity/client readiness timed out; inspect node logs")


def info_snapshot(configs):
    results = []
    for _, port in configs:
        try:
            with HTTP.open(f"http://127.0.0.1:{port}/info", timeout=3) as response:
                results.append(json.load(response))
        except Exception as exc:
            results.append({"error": str(exc)})
    return results


def count_commits(path, protocol):
    text = path.read_text(errors="replace")
    return sum(text.count(tag) for tag in COMMIT_TAGS[protocol])


def client_metrics(paths):
    count, latency_sum = 0, 0.0
    for path in paths:
        for line in path.read_text(errors="replace").splitlines():
            if "[CLIENT_SEC]" not in line:
                continue
            fields = dict(re.findall(r"(\w+)=([^\s]+)", line))
            if 1 <= int(fields["sec"]) <= CLIENT_DURATION_SEC:
                n = int(fields["requests"])
                count += n
                if n:
                    latency_sum += n * float(fields["avg_latency_ms"])
    return {"acked_in_window": count, "throughput_ops_sec": count / CLIENT_DURATION_SEC,
            "mean_latency_ms": latency_sum / count if count else None}


def run_trial(protocol, root, setup_source, load):
    replicas, clients = [], []
    result = {"protocol": protocol, **(load or {}), "status": "failed"}
    try:
        configs = prepare_nodes(root, setup_source)
        save_json(root / "run_config.json", {
            **result, "num_nodes": NUM_NODES, "workload": WORKLOAD,
            "batch_size": SERVER_BATCH_SIZE, "client_duration_sec": CLIENT_DURATION_SEC,
            "client_port": CLIENT_PORT, "send_interval_us": SEND_INTERVAL_US,
            "aggregate_max_in_flight": (load["active_clients"] * load["client_threads"] *
                                        load["max_in_flight"]) if load else None,
        })
        for cfg, _ in configs:
            replicas.append(launch([REPO / "src/stellar-core", "run", "--conf", cfg],
                                   cfg.parent, cfg.parent / "console.log"))
        wait_for_network(replicas, configs, protocol != "scp")
        if protocol == "scp":
            save_json(root / "info-before.json", info_snapshot(configs))
            end = time.monotonic() + SCP_OBSERVATION_SEC
            while time.monotonic() < end:
                ensure_alive(replicas)
                time.sleep(1)
            save_json(root / "info-after.json", info_snapshot(configs))
            result.update(status="diagnostic_only", note=(
                "Native SCP workload creates accounts locally. Ledger advancement alone "
                "does not establish successful workload application; inspect node logs. "
                "Do not compare this run with custom-client throughput."))
        else:
            paths = []
            for i in range(load["active_clients"]):
                path = root / f"client-{i + 1}.log"
                paths.append(path)
                clients.append(launch([
                    root.parent / "shab_client", "127.0.0.1", CLIENT_PORT,
                    load["max_in_flight"], 100000000, SERVER_BATCH_SIZE,
                    CLIENT_DURATION_SEC, SEND_INTERVAL_US, load["client_threads"],
                ], root, path))
            end = time.monotonic() + CLIENT_DURATION_SEC + CLIENT_DRAIN_TIMEOUT_SEC
            while any(p.poll() is None for p in clients) and time.monotonic() < end:
                ensure_alive(replicas)
                time.sleep(0.5)
            ensure_alive(replicas)
            timed_out = any(p.poll() is None for p in clients)
            codes = [p.poll() for p in clients]
            stop_processes(clients)
            metrics = client_metrics(paths)
            per_client = [client_metrics([path])["acked_in_window"] for path in paths]
            commits = [count_commits(cfg.parent / "stellar-core.log", protocol)
                       for cfg, _ in configs]
            result.update(metrics, client_exit_codes=codes, per_client_acked=per_client,
                          per_replica_commit_log_counts=commits)
            if (any(code not in (None, 0) for code in codes) or not all(per_client) or
                    (WORKLOAD != "WORKLOAD_R100" and not all(commits))):
                raise RuntimeError("A client failed, a client received no ACKs, or a replica logged no commits")
            result["status"] = "drain_timeout" if timed_out else "ok"
            if timed_out:
                result["note"] = ("Client did not finish draining within the grace period. "
                                  "A final partial batch can cause this; inspect logs for stalls.")
    except Exception as exc:
        result.update(status="failed", error=str(exc))
    finally:
        stop_processes(clients)
        stop_processes(replicas)
        root.mkdir(parents=True, exist_ok=True)
        save_json(root / "result.json", result)
    return result


def write_summary(session, results):
    save_json(session / "summary.json", results)
    fields = ["protocol", "active_clients", "client_threads", "max_in_flight", "status",
              "acked_in_window", "throughput_ops_sec", "mean_latency_ms", "error", "note"]
    with (session / "summary.csv").open("w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)


def main():
    global REPO, RESULTS_ROOT
    REPO, RESULTS_ROOT = REPO.expanduser().resolve(), RESULTS_ROOT.expanduser().resolve()
    header = REPO / "src/overlay/experiments/ExperimentConfig.h"
    source = REPO / "gcp_setup_stellar_private.sh"
    if not header.is_file() or not source.is_file():
        raise RuntimeError("Set REPO to your updated stellar-core checkout")
    if NUM_NODES < 4 or not PROTOCOLS or not LOAD_POINTS:
        raise ValueError("Use at least four replicas and nonempty protocol/load lists")
    if CLIENT_DURATION_SEC <= 0 or SERVER_BATCH_SIZE <= 0 or BUILD_JOBS <= 0:
        raise ValueError("Duration, batch size and build jobs must be positive")
    if not re.fullmatch(r"WORKLOAD_(A|B|F|W|R0|R10|R20|R25|R40|R50|R60|R75|R80|R100)", WORKLOAD):
        raise ValueError("Unknown WORKLOAD")
    for protocol in PROTOCOLS:
        if protocol != "shabdiz" and protocol not in MODE_FLAGS:
            raise ValueError(f"Unknown protocol: {protocol}")
    for load in LOAD_POINTS:
        if min(load.values()) <= 0:
            raise ValueError("Client counts, threads and in-flight limits must be positive")
        if load["active_clients"] * load["client_threads"] * load["max_in_flight"] < SERVER_BATCH_SIZE:
            raise ValueError("Aggregate in-flight limit must cover at least one server batch")
    # Separate invocations must not race while changing this checkout's build flags.
    lockname = "stellar-local-" + hashlib.sha256(str(REPO).encode()).hexdigest()[:16] + ".lock"
    with (Path(tempfile.gettempdir()) / lockname).open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        session = RESULTS_ROOT / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
        session.mkdir(parents=True)
        original = header.read_bytes()
        setup_source = source.read_text()
        (session / "ExperimentConfig.original.h").write_bytes(original)
        (session / "setup.original.sh").write_text(setup_source)
        command(["git", "rev-parse", "HEAD"], REPO, session / "source.log")
        command(["git", "status", "--short"], REPO, session / "source.log")
        command(["git", "diff", "HEAD"], REPO, session / "source.diff")
        print(f"Results: {session}", flush=True)
        results, changed, interrupted = [], False, False
        try:
            if REGENERATE_BUILD:
                options = CONFIGURE_ARGS
                if options is None:
                    if not (REPO / "config.status").exists():
                        raise RuntimeError("Fresh checkout: set CONFIGURE_ARGS to your configure options")
                    options = shlex.split(subprocess.check_output(
                        ["./config.status", "--config"], cwd=REPO, text=True))
                command(["./autogen.sh"], REPO, session / "bootstrap.log")
                command(["./configure", *options], REPO, session / "bootstrap.log")
            for protocol in PROTOCOLS:
                print(f"\nBuilding {protocol} ...", flush=True)
                folder = session / protocol
                folder.mkdir()
                selected = select_protocol(original.decode(), protocol)
                changed = True
                header.write_text(selected)
                (folder / "ExperimentConfig.h").write_text(selected)
                command(["make", f"-j{BUILD_JOBS}"], REPO, folder / "build.log")
                if protocol != "scp":
                    command([*shlex.split(CXX), "-O2", "-std=c++17", "-pthread",
                             "-I" + str(REPO / "src"), REPO / "shab_client.cpp",
                             "-o", folder / "shab_client"], REPO, folder / "build.log")
                for index, load in enumerate([None] if protocol == "scp" else LOAD_POINTS):
                    label = "native" if load is None else (
                        f"load{index + 1}_c{load['active_clients']}_t{load['client_threads']}_w{load['max_in_flight']}")
                    print(f"Running {protocol}/{label} ...", flush=True)
                    result = run_trial(protocol, folder / label, setup_source, load)
                    results.append(result)
                    write_summary(session, results)
                    print(json.dumps(result), flush=True)
        except KeyboardInterrupt:
            interrupted = True
            print("Interrupted; stopping this script's processes and restoring settings.", flush=True)
            raise
        finally:
            if changed:
                header.write_bytes(original)
                if RESTORE_ORIGINAL_BUILD and not interrupted:
                    print("Restoring original build ...", flush=True)
                    command(["make", f"-j{BUILD_JOBS}"], REPO, session / "restore-build.log")
                else:
                    print("Header restored. Run make before using src/stellar-core manually; "
                          "its binary may still use the last tested protocol.", flush=True)
        print(f"\nFinished. Summary: {session / 'summary.csv'}", flush=True)
        return 1 if any(r["status"] == "failed" for r in results) else 0


def interrupted_by_signal(*_):
    raise KeyboardInterrupt


if __name__ == "__main__":
    signal.signal(signal.SIGTERM, interrupted_by_signal)
    raise SystemExit(main())


Results: /home/tejas/stellar-local-results/20261122-233051-227098

Building pbft_fixed ...
Running pbft_fixed/load1_c1_t1_w150 ...
{"protocol": "pbft_fixed", "active_clients": 1, "client_threads": 1, "max_in_flight": 150, "status": "drain_timeout", "acked_in_window": 2224000, "throughput_ops_sec": 37066.666666666664, "mean_latency_ms": 3.9961590935251796, "client_exit_codes": [null], "per_client_acked": [2224000], "per_replica_commit_log_counts": [22241, 22241, 22241, 22241], "note": "Client did not finish draining within the grace period. A final partial batch can cause this; inspect logs for stalls."}
Running pbft_fixed/load2_c1_t2_w150 ...
{"protocol": "pbft_fixed", "active_clients": 1, "client_threads": 2, "max_in_flight": 150, "status": "drain_timeout", "acked_in_window": 2390376, "throughput_ops_sec": 39839.6, "mean_latency_ms": 7.480023542739718, "client_exit_codes": [null], "per_client_acked": [2390376], "per_replica_commit_log_counts": [23906, 23906, 23906, 23906], "note": "Cl

SystemExit: 0

/home/tejas/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Local failure experiment

In [21]:
"""Local Linux replica-crash experiment. Run this cell independently on a configured checkout."""
import json
import os
import re
import shlex
import signal
import socket
import subprocess
import time
from datetime import datetime
from pathlib import Path
from urllib.error import URLError
from urllib.request import build_opener, ProxyHandler

REPO = Path.cwd() if Path("gcp_setup_stellar_private.sh").is_file() else Path.home() / "stellar-core"
REPO = REPO.expanduser().resolve()
RESULTS_ROOT = Path.home() / "stellar-local-failure-results"
NUM_NODES = 10
PROTOCOLS = ["shabdiz"]
# Other custom-client modes: iths, pbft_fixed, pbft_rotating, tetrabft, simple_it,
# simple_it_bracha, forget_it. Use scp for a native diagnostic without client ACKs.
FAILURE_EVENTS = [(100, 5), (120, 6), (140, 7)]  # (elapsed second, one-based replica). [] = healthy.
CLIENT_DURATION_SEC = 300
LOAD_POINTS = [{"active_clients": 1, "client_threads": 2, "max_in_flight": 100}]
REPETITIONS = 1
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
STARTUP_TIMEOUT_SEC = 360
SEND_INTERVAL_US = 0
BUILD_JOBS = 4
HTTP = build_opener(ProxyHandler({}))

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2) + "\n")


def replace_once(text, pattern, replacement):
    result, count = re.subn(pattern, lambda _: replacement, text, flags=re.MULTILINE)
    if count != 1:
        raise RuntimeError(f"Expected one match for {pattern!r}, found {count}; check repository changes")
    return result


def stop_processes(processes):
    # Only process groups started by THIS script; never pkill other experiments.
    for proc in processes:
        if proc.poll() is None:
            try:
                os.killpg(proc.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
    deadline = time.monotonic() + 5
    for proc in processes:
        try:
            proc.wait(timeout=max(0.01, deadline - time.monotonic()))
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait()


def command(args, cwd, log):
    with log.open("a") as out:
        out.write("\n$ " + shlex.join(map(str, args)) + "\n")
        out.flush()
        proc = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = proc.wait()
            if code:
                raise RuntimeError(f"Command failed ({code}); see {log}")
        finally:
            stop_processes([proc])


def launch(args, cwd, log, config=None):
    with log.open("w") as out:
        if config is not None:
            out.write("# RUN_CONFIG_JSON=" + json.dumps(config) + "\n")
            out.flush()
        return subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)


def check_ports(ports):
    if len(set(ports)) != len(ports):
        raise RuntimeError("Peer, HTTP, and client ports overlap")
    for port in ports:
        with socket.socket() as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("0.0.0.0", port))
            except OSError as exc:
                raise RuntimeError(f"Port {port} is occupied; stop the previous local run") from exc


def local_setup_text(source, root):
    # Reuse your generator, including its seeds, validator lists and port arithmetic.
    text = replace_once(source, r"^STELLAR_CORE=.*$", "STELLAR_CORE=" + shlex.quote(str(REPO / "src/stellar-core")))
    text = replace_once(text, r"^BASE_DIR=.*$", "BASE_DIR=" + shlex.quote(str(root / "nodes")))
    text = replace_once(text, r"^mapfile -t NODE_IPS <.*$",
                        "mapfile -t NODE_IPS < " + shlex.quote(str(root / "localhost_ips.txt")))
    return text.replace('$STELLAR_CORE gen-seed', '"$STELLAR_CORE" gen-seed')


def prepare_nodes(root, setup_source):
    root.mkdir(parents=True)
    (root / "localhost_ips.txt").write_text("127.0.0.1\n" * NUM_NODES)
    setup = root / "setup-local.sh"
    setup.write_text(local_setup_text(setup_source, root))
    command(["bash", setup, "start"], root, root / "setup.log")
    configs, ports = [], []
    for i in range(1, NUM_NODES + 1):
        cfg = root / "nodes" / f"node{i}" / "stellar-core.cfg"
        text = cfg.read_text()
        # All added settings must precede the generator's TOML table headers.
        additions = {
            "SEND_CUSTOM_MESSAGE": "true" if i == 1 else "false",
            "ALLOW_LOCALHOST_FOR_TESTING": "true", "PUBLIC_HTTP_PORT": "false",
            "TARGET_PEER_CONNECTIONS": str(NUM_NODES - 1),
            "MAX_ADDITIONAL_PEER_CONNECTIONS": str(8 * NUM_NODES),
        }
        for key, value in additions.items():
            if re.search(rf"^{key}\s*=", text, re.MULTILINE):
                text = replace_once(text, rf"^{key}\s*=.*$", f"{key}={value}")
            else:
                text = f"{key}={value}\n" + text
        cfg.write_text(text)
        peer = int(re.search(r"^PEER_PORT=(\d+)", text, re.MULTILINE)[1])
        http = int(re.search(r"^HTTP_PORT=(\d+)", text, re.MULTILINE)[1])
        ports.extend([peer, http])
        configs.append((cfg, http))
    check_ports(ports + [CLIENT_PORT])
    # The generator's init branch continues on new-db failure. Check each command here.
    for cfg, _ in configs:
        command([REPO / "src/stellar-core", "new-db", "--conf", cfg], cfg.parent,
                cfg.parent / "init.log")
    return configs


def wait_for_failure_network(replicas, configs, native):
    expected = [set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^"]+)"', cfg.read_text(), re.M))
                for cfg, _ in configs]
    if any(len(keys) != NUM_NODES - 1 for keys in expected):
        raise ValueError("Replica configs have incomplete membership")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
    while time.monotonic() < deadline:
        for i, proc in replicas.items():
            if proc.poll() is not None:
                raise RuntimeError(f"Replica {i} exited during startup; see its console.log")
        ready = True
        for (_, port), keys in zip(configs, expected):
            try:
                with HTTP.open(f"http://127.0.0.1:{port}/peers?fullkeys=true", timeout=1) as response:
                    peers = json.load(response)["authenticated_peers"]
                connected = {p["id"] for direction in ("inbound", "outbound")
                             for p in (peers.get(direction) or [])}
                if not keys <= connected:
                    ready = False
                    break
            except (OSError, URLError):
                ready = False
                break
        if ready:
            log = configs[0][0].parent / "stellar-core.log"
            if native or (log.exists() and f"Client listener started on port {CLIENT_PORT}" in
                          log.read_text(errors="replace")):
                return
        time.sleep(0.3)
    raise TimeoutError("Timed out waiting for authenticated peers or the ingress listener")


def local_failure_trial(session, protocol, repetition, index, load, revision):
    native = protocol == "scp"
    label = "native" if native else f"load{index}_c{load['active_clients']}_t{load['client_threads']}_w{load['max_in_flight']}"
    root = session / protocol / f"repeat{repetition}" / label
    cfg = dict(protocol=protocol, repetition=repetition, load_index=index, num_nodes=NUM_NODES,
               source_commit=revision, failure_schedule=FAILURE_EVENTS, client_duration_sec=CLIENT_DURATION_SEC,
               batch_size=SERVER_BATCH_SIZE, workload=WORKLOAD, **load)
    cfg["aggregate_max_in_flight"] = (None if native else
        load["active_clients"] * load["client_threads"] * load["max_in_flight"])
    env = ["env", f"STELLAR_EXPERIMENT_PROTOCOL={protocol}", f"STELLAR_EXPERIMENT_WORKLOAD={WORKLOAD}",
           f"STELLAR_EXPERIMENT_CLIENT_PORT={CLIENT_PORT}", f"STELLAR_EXPERIMENT_BATCH_SIZE={SERVER_BATCH_SIZE}",
           "STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY=false", "STELLAR_EXPERIMENT_NONPRUNE=false",
           "STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC=2147483647",
           "STELLAR_EXPERIMENT_ITHS_DELTA_MS=1000", "STELLAR_EXPERIMENT_FORGET_IT_DELTA_MS=1000",
           "STELLAR_EXPERIMENT_PBFT_TIMEOUT_MS=2000"]
    replicas, clients, crashed = {}, [], set()
    result = dict(status="failed", config=cfg, failure_events=[])
    print(f"Running {protocol}/{label} (rep {repetition}): {root}", flush=True)
    try:
        configs = prepare_nodes(root, (REPO / "gcp_setup_stellar_private.sh").read_text())
        save_json(root / "run_config.json", cfg)
        for i, (path, _) in enumerate(configs, 1):
            replicas[i] = launch([*env, REPO / "src/stellar-core", "run", "--conf", path],
                                 path.parent, path.parent / "console.log")
        wait_for_failure_network(replicas, configs, native)
        start = time.monotonic()
        epoch = time.time()
        result["observation_start"] = epoch
        result["client_started_at"] = None if native else epoch
        for i in range(load.get("active_clients", 0)):
            folder = root / f"client-{i + 1}"
            folder.mkdir()
            clients.append(launch([*env, session / "shab_client", "127.0.0.1", str(CLIENT_PORT),
                                   str(load["max_in_flight"]), "2000000000", str(SERVER_BATCH_SIZE),
                                   str(CLIENT_DURATION_SEC), str(SEND_INTERVAL_US), str(load["client_threads"])],
                                  folder, folder / "client.log", cfg))
        for offset, node in [*FAILURE_EVENTS, (CLIENT_DURATION_SEC, None)]:
            while True:
                elapsed = time.monotonic() - start
                for i, proc in replicas.items():
                    if i not in crashed and proc.poll() is not None:
                        raise RuntimeError(f"Replica {i} exited unexpectedly ({proc.returncode})")
                for proc in clients:
                    if proc.poll() is not None and (proc.returncode or elapsed < CLIENT_DURATION_SEC):
                        raise RuntimeError(f"Client exited at {elapsed:.2f}s ({proc.returncode})")
                if elapsed >= offset:
                    break
                time.sleep(min(0.1, offset - elapsed))
            if node is not None:
                proc = replicas[node]
                proc.kill()
                sent = time.monotonic() - start
                if proc.wait(timeout=5) != -signal.SIGKILL:
                    raise RuntimeError(f"Could not confirm SIGKILL on replica {node}")
                crashed.add(node)
                result["failure_events"].append(dict(node=node, scheduled_sec=offset, sent_at=epoch + sent))
                print(f"Replica {node} crashed at {sent:.3f}s", flush=True)
        # The client starts its stats clock just after exec. Allow the final
        # one-second record to flush even when outstanding requests never drain.
        flush_end = time.monotonic() + 1.5
        for i, proc in enumerate(clients, 1):
            path = root / f"client-{i}" / "client.log"
            last_tick = f"[CLIENT_SEC] sec={CLIENT_DURATION_SEC} "
            while proc.poll() is None and time.monotonic() < flush_end:
                if last_tick in path.read_text(errors="replace"):
                    break
                time.sleep(0.05)
            if proc.poll() not in (None, 0):
                raise RuntimeError(f"Client {i} exited with code {proc.returncode}")
        result["status"] = "ok"
    except Exception as exc:
        result["error"] = str(exc)
    finally:
        stop_processes(clients)
        stop_processes(replicas.values())
        root.mkdir(parents=True, exist_ok=True)
        save_json(root / "result.json", result)
    print(protocol, result["status"], result.get("error", ""), flush=True)
    return result


def run_local_failures():
    supported = {"shabdiz", "iths", "pbft", "pbft_fixed", "pbft_rotating", "tetrabft",
                 "simple_it", "simple_it_bracha", "forget_it", "scp"}
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - supported:
        raise ValueError("Choose distinct supported protocols")
    if NUM_NODES < 4 or ("forget_it" in PROTOCOLS and (NUM_NODES - 1) % 3):
        raise ValueError("Use at least four replicas and n=3f+1 for Forget-IT")
    times = [t for t, _ in FAILURE_EVENTS]
    nodes = [n for _, n in FAILURE_EVENTS]
    if (times != sorted(set(times)) or any(not 0 < t < CLIENT_DURATION_SEC for t in times)
            or len(nodes) != len(set(nodes)) or len(nodes) > (NUM_NODES - 1) // 3
            or any(type(n) is not int or not 2 <= n <= NUM_NODES for n in nodes)):
        raise ValueError("Choose increasing crash times inside the workload and up to f distinct non-ingress replicas")
    if min(CLIENT_DURATION_SEC, BUILD_JOBS, REPETITIONS, SERVER_BATCH_SIZE) <= 0:
        raise ValueError("Duration, build jobs, repetitions and batch size must be positive")
    if not LOAD_POINTS and set(PROTOCOLS) != {"scp"}:
        raise ValueError("Specify a client load point")
    for load in LOAD_POINTS:
        if (set(load) != {"active_clients", "client_threads", "max_in_flight"}
                or any(type(x) is not int or x < 1 for x in load.values())
                or load["active_clients"] * load["client_threads"] * load["max_in_flight"] < SERVER_BATCH_SIZE):
            raise ValueError(f"Invalid client load: {load}")
    session = RESULTS_ROOT.expanduser().resolve() / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    session.mkdir(parents=True)
    print("Local results:", session, flush=True)
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    print("Checking the configured local build once ...", flush=True)
    command(["make", f"-j{BUILD_JOBS}"], REPO, session / "build.log")
    if set(PROTOCOLS) != {"scp"}:
        command([*shlex.split(os.environ.get("CXX", "g++")), "-O2", "-std=c++17", "-pthread",
                 "-Isrc", "shab_client.cpp", "-o", session / "shab_client"], REPO, session / "build.log")
    results = []
    for protocol in PROTOCOLS:
        for repetition in range(1, REPETITIONS + 1):
            for index, load in ([(0, {})] if protocol == "scp" else enumerate(LOAD_POINTS, 1)):
                results.append(local_failure_trial(session, protocol, repetition, index, load, revision))
                save_json(session / "summary.json", results)
    return session


LOCAL_FAILURE_SESSION = run_local_failures()


Local results: /home/tejas/stellar-local-failure-results/20261124-064332-864446
Checking the configured local build once ...
Running shabdiz/load1_c1_t2_w100 (rep 1): /home/tejas/stellar-local-failure-results/20261124-064332-864446/shabdiz/repeat1/load1_c1_t2_w100
Replica 5 crashed at 100.000s
Replica 6 crashed at 120.000s
Replica 7 crashed at 140.000s
shabdiz ok 


# Local Collection vs Number of Rounds Experiment

In [ ]:
"""Local Shabdiz collection rounds. Run on a configured Linux checkout; no GCP access needed."""
import json
import os
import re
import shlex
import signal
import socket
import subprocess
import time
from datetime import datetime
from pathlib import Path
from urllib.error import URLError
from urllib.request import build_opener, ProxyHandler

REPO = Path.cwd() if Path("gcp_setup_stellar_private.sh").is_file() else Path.home() / "stellar-core"
REPO = REPO.expanduser().resolve()
from importlib import reload
from scripts import collection_rounds

# A notebook kernel can outlive a git pull that changes the measurement schema.
reload(collection_rounds)

RESULTS_ROOT = Path.home() / "stellar-local-collection-rounds"
NUM_NODES = 4
FORCE_COLLECT_AFTER_SEC = 30  # Elapsed from each server process start.
COLLECTION_ROUNDS = [5, 50, 100, 300]  # Add 0 for a no-forcing control.
CLIENT_DURATION_SEC = 40
LOAD_POINTS = [{"active_clients": 1, "client_threads": 2, "max_in_flight": 150}]
REPETITIONS = 1
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
STARTUP_TIMEOUT_SEC = 360
SEND_INTERVAL_US = 0
BUILD_JOBS = 4
HTTP = build_opener(ProxyHandler({}))

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2) + "\n")


def replace_once(text, pattern, replacement):
    result, count = re.subn(pattern, lambda _: replacement, text, flags=re.MULTILINE)
    if count != 1:
        raise RuntimeError(f"Expected one match for {pattern!r}, found {count}; check repository changes")
    return result


def stop_processes(processes):
    # Only process groups started by THIS script; never pkill other experiments.
    for proc in processes:
        if proc.poll() is None:
            try:
                os.killpg(proc.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
    deadline = time.monotonic() + 5
    for proc in processes:
        try:
            proc.wait(timeout=max(0.01, deadline - time.monotonic()))
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            proc.wait()


def command(args, cwd, log):
    with log.open("a") as out:
        out.write("\n$ " + shlex.join(map(str, args)) + "\n")
        out.flush()
        proc = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)
        try:
            code = proc.wait()
            if code:
                raise RuntimeError(f"Command failed ({code}); see {log}")
        finally:
            stop_processes([proc])


def launch(args, cwd, log, config=None):
    with log.open("w") as out:
        if config is not None:
            out.write("# RUN_CONFIG_JSON=" + json.dumps(config) + "\n")
            out.flush()
        return subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=out,
                                stderr=subprocess.STDOUT, start_new_session=True)


def check_ports(ports):
    if len(set(ports)) != len(ports):
        raise RuntimeError("Peer, HTTP, and client ports overlap")
    for port in ports:
        with socket.socket() as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("0.0.0.0", port))
            except OSError as exc:
                raise RuntimeError(f"Port {port} is occupied; stop the previous local run") from exc


def local_setup_text(source, root):
    # Reuse your generator, including its seeds, validator lists and port arithmetic.
    text = replace_once(source, r"^STELLAR_CORE=.*$", "STELLAR_CORE=" + shlex.quote(str(REPO / "src/stellar-core")))
    text = replace_once(text, r"^BASE_DIR=.*$", "BASE_DIR=" + shlex.quote(str(root / "nodes")))
    text = replace_once(text, r"^mapfile -t NODE_IPS <.*$",
                        "mapfile -t NODE_IPS < " + shlex.quote(str(root / "localhost_ips.txt")))
    return text.replace('$STELLAR_CORE gen-seed', '"$STELLAR_CORE" gen-seed')


def prepare_nodes(root, setup_source):
    root.mkdir(parents=True)
    (root / "localhost_ips.txt").write_text("127.0.0.1\n" * NUM_NODES)
    setup = root / "setup-local.sh"
    setup.write_text(local_setup_text(setup_source, root))
    command(["bash", setup, "start"], root, root / "setup.log")
    configs, ports = [], []
    for i in range(1, NUM_NODES + 1):
        cfg = root / "nodes" / f"node{i}" / "stellar-core.cfg"
        text = cfg.read_text()
        # All added settings must precede the generator's TOML table headers.
        additions = {
            "SEND_CUSTOM_MESSAGE": "true" if i == 1 else "false",
            "ALLOW_LOCALHOST_FOR_TESTING": "true", "PUBLIC_HTTP_PORT": "false",
            "TARGET_PEER_CONNECTIONS": str(NUM_NODES - 1),
            "MAX_ADDITIONAL_PEER_CONNECTIONS": str(8 * NUM_NODES),
        }
        for key, value in additions.items():
            if re.search(rf"^{key}\s*=", text, re.MULTILINE):
                text = replace_once(text, rf"^{key}\s*=.*$", f"{key}={value}")
            else:
                text = f"{key}={value}\n" + text
        cfg.write_text(text)
        peer = int(re.search(r"^PEER_PORT=(\d+)", text, re.MULTILINE)[1])
        http = int(re.search(r"^HTTP_PORT=(\d+)", text, re.MULTILINE)[1])
        ports.extend([peer, http])
        configs.append((cfg, http))
    check_ports(ports + [CLIENT_PORT])
    # The generator's init branch continues on new-db failure. Check each command here.
    for cfg, _ in configs:
        command([REPO / "src/stellar-core", "new-db", "--conf", cfg], cfg.parent,
                cfg.parent / "init.log")
    return configs


def wait_for_network(replicas, configs):
    expected = [set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^"]+)"', cfg.read_text(), re.M))
                for cfg, _ in configs]
    if any(len(keys) != NUM_NODES - 1 for keys in expected):
        raise ValueError("Replica configs have incomplete membership")
    deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
    while time.monotonic() < deadline:
        for i, proc in replicas.items():
            if proc.poll() is not None:
                raise RuntimeError(f"Replica {i} exited during startup; see its console.log")
        ready = True
        for (_, port), keys in zip(configs, expected):
            try:
                with HTTP.open(f"http://127.0.0.1:{port}/peers?fullkeys=true", timeout=1) as response:
                    peers = json.load(response)["authenticated_peers"]
                connected = {p["id"] for direction in ("inbound", "outbound")
                             for p in (peers.get(direction) or [])}
                if not keys <= connected:
                    ready = False
                    break
            except (OSError, URLError):
                ready = False
                break
        if ready:
            log = configs[0][0].parent / "stellar-core.log"
            if (log.exists() and f"Client listener started on port {CLIENT_PORT}" in
                          log.read_text(errors="replace")):
                return
        time.sleep(0.3)
    raise TimeoutError("Timed out waiting for authenticated peers or the ingress listener")


def local_collection_trial(session, rounds, repetition, index, load, revision):
    label = f"load{index}_c{load['active_clients']}_t{load['client_threads']}_w{load['max_in_flight']}"
    root = session / f"rounds{rounds}" / f"repeat{repetition}" / label
    cfg = dict(protocol="shabdiz", repetition=repetition, num_nodes=NUM_NODES, source_commit=revision,
               collection_rounds=rounds, force_collect_after_sec=FORCE_COLLECT_AFTER_SEC,
               client_duration_sec=CLIENT_DURATION_SEC, batch_size=SERVER_BATCH_SIZE, workload=WORKLOAD, **load)
    env = ["env", "STELLAR_EXPERIMENT_PROTOCOL=shabdiz", f"STELLAR_EXPERIMENT_WORKLOAD={WORKLOAD}",
           f"STELLAR_EXPERIMENT_CLIENT_PORT={CLIENT_PORT}", f"STELLAR_EXPERIMENT_BATCH_SIZE={SERVER_BATCH_SIZE}",
           "STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY=false", "STELLAR_EXPERIMENT_NONPRUNE=false",
           "STELLAR_EXPERIMENT_COLLECTION_START_VIEW=0",
           f"STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC={FORCE_COLLECT_AFTER_SEC}",
           f"STELLAR_EXPERIMENT_MAX_COLLECT_ATTEMPTS={rounds}"]
    replicas, clients = {}, []
    result = dict(status="failed", config=cfg)
    print(f"Running {rounds} rounds after {FORCE_COLLECT_AFTER_SEC}s of server uptime: {root}", flush=True)
    try:
        configs = prepare_nodes(root, (REPO / "gcp_setup_stellar_private.sh").read_text())
        save_json(root / "run_config.json", cfg)
        for i, (path, _) in enumerate(configs, 1):
            replicas[i] = launch([*env, REPO / "src/stellar-core", "run", "--conf", path],
                                 path.parent, path.parent / "console.log")
        wait_for_network(replicas, configs)
        start = time.monotonic()
        epoch = time.time()
        result["client_started_at"] = epoch
        for i in range(load.get("active_clients", 0)):
            folder = root / f"client-{i + 1}"
            folder.mkdir()
            clients.append(launch([*env, session / "shab_client", "127.0.0.1", str(CLIENT_PORT),
                                   str(load["max_in_flight"]), "2000000000", str(SERVER_BATCH_SIZE),
                                   str(CLIENT_DURATION_SEC), str(SEND_INTERVAL_US), str(load["client_threads"])],
                                  folder, folder / "client.log", cfg))
        end = start + CLIENT_DURATION_SEC
        while time.monotonic() < end:
            if any(p.poll() is not None for p in replicas.values()):
                raise RuntimeError("Replica exited during workload; inspect its console.log")
            if any(p.poll() is not None for p in clients):
                raise RuntimeError("Client exited before the workload ended")
            time.sleep(min(0.1, max(0, end - time.monotonic())))
        # The client starts its stats clock just after exec. Allow the final
        # one-second record to flush even when outstanding requests never drain.
        flush_end = time.monotonic() + 1.5
        for i, proc in enumerate(clients, 1):
            path = root / f"client-{i}" / "client.log"
            last_tick = f"[CLIENT_SEC] sec={CLIENT_DURATION_SEC} "
            while proc.poll() is None and time.monotonic() < flush_end:
                if last_tick in path.read_text(errors="replace"):
                    break
                time.sleep(0.05)
            if proc.poll() not in (None, 0):
                raise RuntimeError(f"Client {i} exited with code {proc.returncode}")
        result["status"] = "ok"
    except Exception as exc:
        result["error"] = str(exc)
    finally:
        stop_processes(clients)
        stop_processes(replicas.values())
        root.mkdir(parents=True, exist_ok=True)
        if result["status"] == "ok":
            try:
                result.update(collection_rounds.observed_rounds(root.glob("nodes/node*/stellar-core.log"), cfg, result["client_started_at"]))
            except (OSError, ValueError) as exc:
                result.update(status="failed", error=str(exc))
        save_json(root / "result.json", result)
    print(rounds, result["status"], result.get("error", ""), flush=True)
    return result


def run_local_collection_rounds():
    collection_rounds.check_round_settings(FORCE_COLLECT_AFTER_SEC, COLLECTION_ROUNDS)
    if not 4 <= NUM_NODES <= 256:
        raise ValueError("Shabdiz requires 4..256 replicas")
    if min(CLIENT_DURATION_SEC, BUILD_JOBS, REPETITIONS, SERVER_BATCH_SIZE) <= 0:
        raise ValueError("Duration, build jobs, repetitions and batch size must be positive")
    if not LOAD_POINTS:
        raise ValueError("Specify a client load point")
    for load in LOAD_POINTS:
        if (set(load) != {"active_clients", "client_threads", "max_in_flight"}
                or any(type(x) is not int or x < 1 for x in load.values())
                or load["active_clients"] * load["client_threads"] * load["max_in_flight"] < SERVER_BATCH_SIZE):
            raise ValueError(f"Invalid client load: {load}")
    session = RESULTS_ROOT.expanduser().resolve() / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    session.mkdir(parents=True)
    print("Local results:", session, flush=True)
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    print("Checking the configured local build once ...", flush=True)
    command(["make", f"-j{BUILD_JOBS}"], REPO, session / "build.log")
    command([*shlex.split(os.environ.get("CXX", "g++")), "-O2", "-std=c++17", "-pthread",
                 "-Isrc", "shab_client.cpp", "-o", session / "shab_client"], REPO, session / "build.log")
    results = []
    for rounds in COLLECTION_ROUNDS:
        for repetition in range(1, REPETITIONS + 1):
            for index, load in enumerate(LOAD_POINTS, 1):
                results.append(local_collection_trial(session, rounds, repetition, index, load, revision))
                save_json(session / "summary.json", results)
    return session


LOCAL_COLLECTION_ROUNDS_SESSION = run_local_collection_rounds()


# GCP Client Load: Throughput vs Latency (All Protocols)

In [15]:
"""GCP throughput versus ACK latency; run this cell after editing the settings.
Builds and runs only when this cell is executed. No automatic git commit/push.
Checks the image's configured build once; all protocols/sizes reuse the binary.
Creates/prepares VMs in parallel, then sweeps protocols and loads on the same fleet.
Pins one source commit and preserves raw logs.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time
import uuid

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_DIR = Path.home() / "work/experiments/gcp-client-load/ClientThputLatency"
# Fixed output folder. Rerunning the same protocol/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False saves only stellar-core.log and client.log.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
# Both modes run ALL configured trials into RESULTS_DIR; no automatic timestamp folder.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-load"
INSTANCE_PREFIX = "tsm-sc-load"         # Original fleet names; using e2-standard-4 VMs.
NUM_NODES = 10                         # Forget-IT requires 3f+1: 4, 7, 10, ...
REPLICA_ZONES = ["us-central1-c"] * NUM_NODES
CLIENT_ZONES = ["us-central1-c", "us-central1-c"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # make -j4 in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOLS = [ "shabdiz"]
# PROTOCOLS = ["tetrabft"]

# simple_it = Opt-RBC, non-speculative. Optionally add "simple_it_bracha".
# SCP has a different native workload/ACK interface and is not comparable here.
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 210
MEASUREMENT_START_SEC = 150              # ACK completion window [30, CLIENT_DURATION_SEC).
MEASUREMENT_END_SEC = CLIENT_DURATION_SEC
STARTUP_TIMEOUT_SEC = 270
CLIENT_START_LEAD_SEC = 45              # Common wall-clock start across client VMs.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
ITHS_BLOG_DELTA_MS = 1000               # Explicit: 2 Delta wait EACH sequential batch.
PBFT_TIMEOUT_MS = 2000                  # Initial timeout for stalled PBFT work.
FORGET_IT_DELTA_MS = 1000               # Recovery timing; preserves current default.
SHABDIZ_FAST_PATH_ONLY = False          # True: memory-only block archive; keep stellar-core.log.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 1, "max_in_flight": 100},  # 100 total
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 75},   # 150 total
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 200 total
    {"active_clients": 2, "client_threads": 2, "max_in_flight": 75},   # 300 total
    {"active_clients": 2, "client_threads": 2, "max_in_flight": 100},  # 400 total
    {"active_clients": 2, "client_threads": 2, "max_in_flight": 125},  # 500 total

    # {"active_clients": 2, "client_threads": 2, "max_in_flight": 150},  # 600
    # {"active_clients": 2, "client_threads": 2, "max_in_flight": 200},  # 800
    # {"active_clients": 2, "client_threads": 2, "max_in_flight": 250},  # 1000

]
# By default VMs remain running for reuse. Deletion applies only to this run\'s fleet.
# ------------------------------------------------------

COMMIT_TAGS = {"shabdiz": ["[FAST COMMITTED]"],
               "iths": ["[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"],
               "pbft_fixed": ["[PBFT COMMITTED-LOCAL]"], "pbft_rotating": ["[PBFT COMMITTED-LOCAL]"],
               "pbft": ["[PBFT COMMITTED-LOCAL]"], "tetrabft": ["[TETRA COMMITTED]"],
               "simple_it": ["[SIMPLE-IT COMMITTED]"],
               "simple_it_bracha": ["[SIMPLE-IT COMMITTED]"],
               "forget_it": ["[FORGET-IT COMMITTED]"]}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': WORKLOAD,
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_ITHS_DELTA_MS': str(ITHS_BLOG_DELTA_MS),
        'STELLAR_EXPERIMENT_PBFT_TIMEOUT_MS': str(PBFT_TIMEOUT_MS),
        'STELLAR_EXPERIMENT_FORGET_IT_DELTA_MS': str(FORGET_IT_DELTA_MS),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': str(SHABDIZ_FAST_PATH_ONLY).lower(),
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': '2147483647',
    }


def runtime_command(args, protocol):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOLS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-client-load", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    # Each retry gets a new directory; preserve the failed attempt logs.
    trial = REMOTE_SESSION + "/runs/" + label + "/attempt-" + uuid.uuid4().hex
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        parallel(stop_vm_processes, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol)),
                log=folder / f"node-{vm['index']}-init.log")
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = start_at + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        if result['status'] in ('ok', 'drain_timeout'):
            def count_commits(vm):
                with (logs_folder / f"node-{vm['index']}" / 'stellar-core.log').open(errors='replace') as log:
                    return sum(line.count(tag) for line in log for tag in COMMIT_TAGS[protocol])
            counts = parallel(count_commits, nodes)
            result['replica_commits'] = counts
            if not all(counts):
                result.update(status='failed', error='A replica logged no commits')
        result['timings_sec'] = timings
        save_json(folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, load, repetition, index):
    return (f"{protocol}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the GCP client-load plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def main_gcp_load(work_dir):
    global REMOTE_SESSION, REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if NUM_NODES < 4 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and at least four replicas")
    if 'forget_it' in PROTOCOLS and (NUM_NODES - 1) % 3:
        raise ValueError("Forget-IT requires n=3f+1 (4, 7, 10, ...); 8 is unsupported")
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - COMMIT_TAGS.keys():
        raise ValueError("Choose distinct custom-client protocols from COMMIT_TAGS; SCP is diagnostic only")
    if not (0 <= MEASUREMENT_START_SEC < MEASUREMENT_END_SEC <= CLIENT_DURATION_SEC):
        raise ValueError("Invalid measurement window")
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE, ITHS_BLOG_DELTA_MS, FORGET_IT_DELTA_MS) <= 0:
        raise ValueError("Counts, batch size, and Delta must be positive")
    if not re.fullmatch(r'WORKLOAD_(W|R0)', WORKLOAD):
        raise ValueError("This comparison uses write-only transactions; choose WORKLOAD_W or WORKLOAD_R0")
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Only remote working paths need unique IDs. Local results use the fixed input.
    session = Path(work_dir)
    REMOTE_SESSION = REMOTE_ROOT + '/' + session_id
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'workload': WORKLOAD, 'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'measurement_start_sec': MEASUREMENT_START_SEC, 'measurement_end_sec': MEASUREMENT_END_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'pbft_timeout_ms': PBFT_TIMEOUT_MS,
              'iths_blog_delta_ms': ITHS_BLOG_DELTA_MS, 'forget_it_delta_ms': FORGET_IT_DELTA_MS,
              'shabdiz_fast_path_only': SHABDIZ_FAST_PATH_ONLY, 'forced_collection': False,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocols': PROTOCOLS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(protocol, repetition, index, load) for protocol in PROTOCOLS
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    print(f'IT-HS Delta={ITHS_BLOG_DELTA_MS} ms: its per-batch proposal wait is {2*ITHS_BLOG_DELTA_MS} ms.')
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_SESSION]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before running IT-HS/scheduled clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOLS[0]
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for protocol in PROTOCOLS:
            protocol_trials = [trial for trial in trials if trial[0] == protocol]
            print(f'{protocol}: reusing common binaries; selecting runtime settings.', flush=True)
            for _, repetition, index, load in protocol_trials:
                result = run_trial(vms, builder, art, protocol, load, repetition, index, session, common, client_art)
                results.append(result)
                save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; VMs kept. Set RESUME_SESSION=True to rerun all trials with existing builds.')
    else:
        finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_load():
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Config archives, setup logs, timing files, and status JSON are temporary.
    # The fixed output contains only per-trial replica/client logs and optional builds.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-load-') as work_dir:
        return main_gcp_load(work_dir)


GCP_LOAD_SESSION = run_gcp_load()



Results: /home/tejas/work/experiments/gcp-client-load/ClientThputLatency
6 trials; 0.3 hours of workload, plus builds and setup; pkill at workload end.
IT-HS Delta=1000 ms: its per-batch proposal wait is 2000 ms.
Creating tsm-sc-load-node-01 in us-central1-c...
Creating tsm-sc-load-node-02 in us-central1-c...
Creating tsm-sc-load-node-03 in us-central1-c...
Creating tsm-sc-load-node-04 in us-central1-c...
Creating tsm-sc-load-node-05 in us-central1-c...
Creating tsm-sc-load-node-06 in us-central1-c...
Creating tsm-sc-load-node-07 in us-central1-c...
Creating tsm-sc-load-node-08 in us-central1-c...
Creating tsm-sc-load-node-09 in us-central1-c...
Creating tsm-sc-load-node-10 in us-central1-c...
Creating tsm-sc-load-client-01 in us-central1-c...
Creating tsm-sc-load-client-02 in us-central1-c...
tsm-sc-load-node-01: created
tsm-sc-load-node-08: created
tsm-sc-load-node-04: created
tsm-sc-load-node-02: created
tsm-sc-load-node-10: created
tsm-sc-load-client-02: created
tsm-sc-load-node-07

# Main num nodes experiment

In [10]:
"""GCP throughput and ACK latency versus replica count.
Run this cell independently; it does not run or depend on the client-load cell.
Create the largest fleet in parallel, build once, run every protocol at each
size, then delete surplus replicas before moving to the next smaller size.
Only this cell's replica/client names are managed. No automatic git commit/push.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time
import uuid

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Updated local stellar-core checkout; HEAD must be pushed.
RESULTS_DIR = Path.home() / "work/experiments/gcp-num-nodes/SystemSize"
# Fixed directory: n49/shabdiz/repeat1/load1_.../{node-*,client-*}/*.log.
KEEP_BUILD_LOGS = True
RESUME_SESSION = False                 # True reuses VM source, but reruns every configured trial.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"
REMOTE_ROOT = "/home/tejas/stellar-gcp-size"
INSTANCE_PREFIX = "tsm-sc-size"         # Separate from the e2-standard-4 client-load fleet.
# Nearest 3f+1 sizes to 48, 32, 16, 8, 4, so Forget-IT runs at EVERY size too.
NUM_NODES_LIST = [49, 31, 16, 7, 4]      # Strictly descending; replica indices/zones stay fixed.\
# NUM_NODES_LIST = [49, 31]      # Strictly descending; replica indices/zones stay fixed.

# One zone for each replica in the LARGEST fleet. Smaller sizes use its prefix.
REPLICA_ZONES = ["us-central1-c"] * max(NUM_NODES_LIST)
CLIENT_ZONES = ["us-central1-c", "us-central1-c"]  # Additional VMs, retained across all sizes.
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50
CREATE_MISSING_VMS = True
DELETE_ALL_AFTER_FINAL_RUN = True       # After successful n=4 runs, delete the remaining 4 + 2 VMs.
# Surplus replicas are always deleted after all protocols at a size finish successfully.
USE_IAP = False
BUILD_JOBS = 2                         # Incremental make -j2 once per replica; no autogen/configure.
MAX_PARALLEL_WORKERS = 48
STREAM_BUILD_OUTPUT = False
SOURCE_COMMIT = None                   # None pins the local HEAD once for the whole sweep.

PROTOCOLS = [ "forget_it", "tetrabft", "simple_it"]
# Selected throughput/latency operating points from the client-load experiment.
# These remain fixed per protocol across sizes, rather than claiming peak throughput at every size.
PROTOCOL_LOADS = {
    # Matched 400-request comparison point. Tune from the PBFT load sweeps.
    "pbft_fixed": {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},# 200
    "pbft_rotating": {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},# 200
    "shabdiz":   {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 200
    "tetrabft":  {"active_clients": 2, "client_threads": 2, "max_in_flight": 150},  # 600
    "simple_it": {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 200
    "forget_it": {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},   # 200
    "iths": {"active_clients": 1, "client_threads": 1, "max_in_flight": 100},   # 100
}
# To add another custom-client protocol, supply its measured load above too.
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 210
MEASUREMENT_START_SEC = 150
MEASUREMENT_END_SEC = CLIENT_DURATION_SEC
STARTUP_TIMEOUT_SEC = 270
CLIENT_START_LEAD_SEC = 45
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Set >=3 for repeatability/error bars.
ITHS_BLOG_DELTA_MS = 1000
PBFT_TIMEOUT_MS = 2000                  # Initial timeout for stalled PBFT work.
FORGET_IT_DELTA_MS = 1000
SHABDIZ_FAST_PATH_ONLY = False          # True: memory-only block archive; keep stellar-core.log.
# ------------------------------------------------------

COMMIT_TAGS = {"shabdiz": ["[FAST COMMITTED]"],
               "iths": ["[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"],
               "pbft_fixed": ["[PBFT COMMITTED-LOCAL]"], "pbft_rotating": ["[PBFT COMMITTED-LOCAL]"],
               "pbft": ["[PBFT COMMITTED-LOCAL]"], "tetrabft": ["[TETRA COMMITTED]"],
               "simple_it": ["[SIMPLE-IT COMMITTED]"],
               "simple_it_bracha": ["[SIMPLE-IT COMMITTED]"],
               "forget_it": ["[FORGET-IT COMMITTED]"]}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': WORKLOAD,
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_ITHS_DELTA_MS': str(ITHS_BLOG_DELTA_MS),
        'STELLAR_EXPERIMENT_PBFT_TIMEOUT_MS': str(PBFT_TIMEOUT_MS),
        'STELLAR_EXPERIMENT_FORGET_IT_DELTA_MS': str(FORGET_IT_DELTA_MS),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': str(SHABDIZ_FAST_PATH_ONLY).lower(),
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': '2147483647',
    }


def runtime_command(args, protocol):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOLS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-client-load", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    # Each retry gets a new directory; preserve the failed attempt logs.
    trial = REMOTE_SESSION + "/runs/" + label + "/attempt-" + uuid.uuid4().hex
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        parallel(stop_vm_processes, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol)),
                log=folder / f"node-{vm['index']}-init.log")
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = start_at + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        if result['status'] in ('ok', 'drain_timeout'):
            def count_commits(vm):
                with (logs_folder / f"node-{vm['index']}" / 'stellar-core.log').open(errors='replace') as log:
                    return sum(line.count(tag) for line in log for tag in COMMIT_TAGS[protocol])
            counts = parallel(count_commits, nodes)
            result['replica_commits'] = counts
            if not all(counts):
                result.update(status='failed', error='A replica logged no commits')
        result['timings_sec'] = timings
        save_json(folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, load, repetition, index):
    return (f"n{NUM_NODES}/{protocol}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the Main num nodes experiment plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def validate_size_settings():
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if (not NUM_NODES_LIST or any(type(n) is not int or n < 4 for n in NUM_NODES_LIST)
            or any(a <= b for a, b in zip(NUM_NODES_LIST, NUM_NODES_LIST[1:]))):
        raise ValueError('NUM_NODES_LIST must contain distinct sizes >=4 in strictly descending order')
    if len(REPLICA_ZONES) != NUM_NODES_LIST[0]:
        raise ValueError('Specify one REPLICA_ZONES entry per replica in the largest fleet')
    if 'forget_it' in PROTOCOLS and any((n - 1) % 3 for n in NUM_NODES_LIST):
        raise ValueError('Forget-IT requires n=3f+1: use [49, 31, 16, 7, 4] for the common sweep')
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - COMMIT_TAGS.keys():
        raise ValueError('Choose distinct custom-client protocols; SCP uses a different workload')
    if set(PROTOCOLS) - PROTOCOL_LOADS.keys():
        raise ValueError('Supply PROTOCOL_LOADS for every selected protocol')
    if not (0 <= MEASUREMENT_START_SEC < MEASUREMENT_END_SEC <= CLIENT_DURATION_SEC):
        raise ValueError('Invalid measurement window')
    counts = (REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE, MAX_PARALLEL_WORKERS)
    if any(type(v) is not int or v <= 0 for v in counts):
        raise ValueError('Repetitions, build jobs, batch size, and parallel workers must be positive integers')
    if min(ITHS_BLOG_DELTA_MS, FORGET_IT_DELTA_MS) <= 0:
        raise ValueError('Delta must be positive')
    if not re.fullmatch(r'WORKLOAD_(W|R0)', WORKLOAD):
        raise ValueError('Use write-only transactions: WORKLOAD_W or WORKLOAD_R0')
    for protocol in PROTOCOLS:
        load = PROTOCOL_LOADS[protocol]
        if (set(load) != {'active_clients', 'client_threads', 'max_in_flight'}
                or any(type(v) is not int or v <= 0 for v in load.values())):
            raise ValueError(f'Invalid load for {protocol}: {load}')
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError(f'{protocol}: insufficient client VMs or concurrency for one full batch')


def shrink_fleet(vms, next_size, session):
    # Client VMs have separate names/indices, so every client survives all shrinks.
    surplus = [v for v in vms if v['role'] == 'node' and v['index'] > next_size]
    print(f'Deleting {len(surplus)} surplus replicas before n={next_size}: ' +
          ', '.join(v['name'] for v in surplus), flush=True)
    parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
             surplus)
    retained = [v for v in vms if v['role'] == 'client' or v['index'] <= next_size]
    saved = json.loads((session / 'fleet.json').read_text())
    save_json(session / 'fleet.json', {**saved, 'vms': retained})
    return retained


def cleanup_size_fleet(vms):
    # Deletion may have partially succeeded before an exception/interrupt.
    # Do not attempt SSH to deleted VMs, and still try every surviving VM.
    existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                json.loads(cloud('instances', 'list', '--format=json'))}
    surviving = [v for v in vms if (v['name'], v['zone']) in existing]
    parallel(lambda vm: stop_vm_processes(vm, include_builds=True), surviving)


def main_gcp_sizes(work_dir):
    global REMOTE_SESSION, REMOTE_BUILD, NUM_NODES
    validate_size_settings()           # Validate everything before creating/deleting VMs.
    NUM_NODES = NUM_NODES_LIST[0]
    repo = REPO.expanduser().resolve()
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    session = Path(work_dir)
    session.mkdir(parents=True, exist_ok=True)
    REMOTE_SESSION = REMOTE_ROOT + '/' + session_id
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'experiment': 'num_nodes', 'session': session_id,
              'source_commit': sha, 'workload': WORKLOAD, 'batch_size': SERVER_BATCH_SIZE,
              'client_duration_sec': CLIENT_DURATION_SEC,
              'measurement_start_sec': MEASUREMENT_START_SEC, 'measurement_end_sec': MEASUREMENT_END_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'pbft_timeout_ms': PBFT_TIMEOUT_MS,
              'iths_blog_delta_ms': ITHS_BLOG_DELTA_MS, 'forget_it_delta_ms': FORGET_IT_DELTA_MS,
              'shabdiz_fast_path_only': SHABDIZ_FAST_PATH_ONLY, 'forced_collection': False,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocols': PROTOCOLS, 'protocol_loads': PROTOCOL_LOADS,
                'num_nodes_list': NUM_NODES_LIST, 'repetitions': REPETITIONS,
                'instance_prefix': INSTANCE_PREFIX, 'resume_session': RESUME_SESSION,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN}
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    print('Replica counts:', NUM_NODES_LIST, '; additional client VMs:', len(CLIENT_ZONES), flush=True)
    print('Loads:', {p: math.prod(PROTOCOL_LOADS[p].values()) for p in PROTOCOLS}, flush=True)
    total = len(NUM_NODES_LIST) * len(PROTOCOLS) * REPETITIONS
    print(f'{total} trials; {total * CLIENT_DURATION_SEC / 3600:.2f} hours of workload, '
          'plus startup/builds/log collection. pkill at workload end.', flush=True)
    results = []
    save_json(session / 'results.json', results)
    vms = fleet(session)               # Create ONLY ONCE, with the largest replica count.
    try:
        parallel(wait_ssh, vms)
        common['resolved_image'] = json.loads((session / 'fleet.json').read_text())['image']
        print('Stopping stale replicas, clients, and builds on the largest fleet...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_SESSION]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before scheduled clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role'] == 'node')
        if RESUME_SESSION:
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ; rerun with RESUME_SESSION=False')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            def prepare_vm(vm):
                folder = session / 'preparation' / vm['name']
                folder.mkdir(parents=True, exist_ok=True)
                prepare_build(vm, sha, folder)
            print('Preparing the pinned source once on every VM...', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        def build_common(vm):
            folder = session / 'builds' / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        print(f'Checking common binaries once, make -j{BUILD_JOBS} per replica...', flush=True)
        parallel(build_common, vms)
        art, client_art = IMAGE_REPO + '/src', IMAGE_REPO
        for size_index, num_nodes in enumerate(NUM_NODES_LIST):
            NUM_NODES = num_nodes      # Config membership only; no header edits/recompilation.
            nodes = [v for v in vms if v['role'] == 'node']
            if len(nodes) != num_nodes:
                raise RuntimeError(f'Expected {num_nodes} replicas, got {len(nodes)}')
            run_common = {**common, 'num_nodes': num_nodes,
                          'replica_zones': [v['zone'] for v in nodes]}
            print(f'\n=== {num_nodes} replicas + {len(CLIENT_ZONES)} client VMs ===', flush=True)
            size_results = []
            for protocol in PROTOCOLS:
                load = dict(PROTOCOL_LOADS[protocol])
                print(f'{protocol}: {math.prod(load.values())} concurrent requests; reusing binaries.', flush=True)
                for repetition in range(1, REPETITIONS + 1):
                    result = run_trial(vms, builder, art, protocol, load, repetition, 1,
                                       session, run_common, client_art)
                    result['num_nodes'] = num_nodes
                    size_results.append(result)
                    results.append(result)
                    save_json(session / 'results.json', results)
            failed = [r for r in size_results if r['status'] != 'ok']
            if failed:
                raise RuntimeError(f'{len(failed)} trials failed at n={num_nodes}; '
                                   'logs collected, fleet kept for inspection; smaller sizes were not started')
            if size_index + 1 < len(NUM_NODES_LIST):
                vms = shrink_fleet(vms, NUM_NODES_LIST[size_index + 1], session)
    finally:
        cleanup_size_fleet(vms)
    finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_sizes():
    validate_size_settings()
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Temporary setup/status files disappear; retain only replica/client logs and optional build logs.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-size-') as work_dir:
        return main_gcp_sizes(work_dir)


GCP_SIZE_SESSION = run_gcp_sizes()


Results: /home/tejas/work/experiments/gcp-num-nodes/SystemSize
Replica counts: [49, 31, 16, 7, 4] ; additional client VMs: 2
Loads: {'forget_it': 200, 'tetrabft': 600, 'simple_it': 200}
15 trials; 0.33 hours of workload, plus startup/builds/log collection. pkill at workload end.
Creating tsm-sc-size-node-01 in us-central1-c...
Creating tsm-sc-size-node-02 in us-central1-c...
Creating tsm-sc-size-node-03 in us-central1-c...
Creating tsm-sc-size-node-04 in us-central1-c...
Creating tsm-sc-size-node-05 in us-central1-c...
Creating tsm-sc-size-node-06 in us-central1-c...
Creating tsm-sc-size-node-07 in us-central1-c...
Creating tsm-sc-size-node-08 in us-central1-c...
Creating tsm-sc-size-node-09 in us-central1-c...
Creating tsm-sc-size-node-10 in us-central1-c...
Creating tsm-sc-size-node-11 in us-central1-c...
Creating tsm-sc-size-node-12 in us-central1-c...
Creating tsm-sc-size-node-13 in us-central1-c...
Creating tsm-sc-size-node-14 in us-central1-c...
Creating tsm-sc-size-node-15 in us

# Failure Experiment

In [22]:
"""Run independently. Each trial starts ten healthy replicas, then crashes up to
three replica processes during the same client workload. Restart the full fleet for
each protocol/load/repetition. Logs and actual crash times remain in RESULTS_DIR.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time
import uuid

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_DIR = Path.home() / "work/experiments/gcp-node-failure"
# Fixed output folder. Rerunning the same protocol/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False saves only stellar-core.log and client.log.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
# Both modes run ALL configured trials into RESULTS_DIR; no automatic timestamp folder.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-failure"
INSTANCE_PREFIX = "tsm-sc-failure"      # Dedicated failure-experiment fleet.
NUM_NODES = 10                         # Forget-IT requires 3f+1: 4, 7, 10, ...
REPLICA_ZONES = ["us-central1-c"] * NUM_NODES
CLIENT_ZONES = ["us-central1-c"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # make -j2 in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOLS = ["shabdiz"]
# "pbft" is an alias for "pbft_fixed".
# (Seconds from scheduled client start, one-based replica index). Use [] for a healthy control.
FAILURE_EVENTS = [(100, 5), (120, 6), (140, 7)]

# simple_it uses Opt-RBC; simple_it_bracha selects the Bracha-RBC variant.
# Add "scp" for a native replica-log diagnostic on the same crash schedule.
# SCP has no custom-client ACK interface; it runs once per repetition without a load point.
WORKLOAD = "WORKLOAD_W"
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 210
MEASUREMENT_START_SEC = 0                # Include healthy operation and all failure stages.
MEASUREMENT_END_SEC = CLIENT_DURATION_SEC
STARTUP_TIMEOUT_SEC = 270
CLIENT_START_LEAD_SEC = 45              # Common wall-clock start across client VMs.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
ITHS_BLOG_DELTA_MS = 1000               # Explicit: 2 Delta wait EACH sequential batch.
PBFT_TIMEOUT_MS = 2000                  # Initial timeout for stalled PBFT work.
FORGET_IT_DELTA_MS = 1000               # Recovery timing; preserves current default.
SHABDIZ_FAST_PATH_ONLY = False          # True: memory-only block archive; keep stellar-core.log.
# Preserve the original failure experiment's aggregate concurrency of 200.
LOAD_POINTS = [{"active_clients": 1, "client_threads": 2, "max_in_flight": 100}]
# By default VMs remain running for reuse. Deletion applies only to this run\'s fleet.
# ------------------------------------------------------

SUPPORTED_PROTOCOLS = {"shabdiz", "iths", "pbft", "pbft_fixed", "pbft_rotating",
                       "tetrabft", "simple_it", "simple_it_bracha", "forget_it", "scp"}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': WORKLOAD,
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_ITHS_DELTA_MS': str(ITHS_BLOG_DELTA_MS),
        'STELLAR_EXPERIMENT_PBFT_TIMEOUT_MS': str(PBFT_TIMEOUT_MS),
        'STELLAR_EXPERIMENT_FORGET_IT_DELTA_MS': str(FORGET_IT_DELTA_MS),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': str(SHABDIZ_FAST_PATH_ONLY).lower(),
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': '2147483647',
    }


def runtime_command(args, protocol):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOLS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-node-failure", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def inject_failures(nodes, trial, start_at, cancel, events):
    # Start SSH ahead of each deadline, then use the target VM's synchronized clock.
    # Only signal the replica PID from this trial and confirm it exited.
    for offset, index in FAILURE_EVENTS:
        at = start_at + offset
        if cancel.wait(max(0, at - time.time() - 10)):
            return
        vm = nodes[index - 1]
        root = trial + f"/nodes/node{index}"
        code = f"""
import json, os, pathlib, signal, time
root = pathlib.Path({root!r})
time.sleep(max(0, {at!r} - time.time()))
pid = int((root/'pid').read_text())
proc = pathlib.Path('/proc') / str(pid)
if (proc/'cwd').resolve(strict=True) != root or (proc/'comm').read_text().strip() != 'stellar-core':
    raise RuntimeError('Recorded PID is no longer the trial replica')
os.kill(pid, signal.SIGKILL)
sent_at = time.time()
deadline = time.monotonic() + 5
while True:
    try:
        if (proc/'stat').read_text().rsplit(')', 1)[1].split()[0] == 'Z':
            break
    except FileNotFoundError:
        break
    if time.monotonic() >= deadline:
        raise RuntimeError('Replica still alive after SIGKILL')
    time.sleep(0.05)
print(json.dumps(sent_at))
"""
        sent_at = json.loads(python_remote(vm, code, timeout=45))
        events.append(dict(node=index, scheduled_sec=offset, sent_at=sent_at))
        print(f"Replica {index} crashed at {sent_at - start_at:.2f}s", flush=True)


def run_trial(vms, builder, art, protocol, load, repetition, load_index, session, common, client_art=None):
    native = protocol == 'scp'
    label = (f"scp/repeat{repetition}/native" if native
             else trial_label(protocol, load, repetition, load_index))
    schedule_id = '_'.join(f'r{node}at{sec}' for sec, node in FAILURE_EVENTS) or 'healthy'
    label += '_' + schedule_id
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # Invalidate previous status before replacing logs, including on interruption.
    save_json(logs_folder / "result.json", {"status": "running", "protocol": protocol})
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    # Give each remote attempt its own working directory.
    trial = REMOTE_SESSION + "/runs/" + label + "/attempt-" + uuid.uuid4().hex
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load.get('active_clients', 0)]
    config = {**common, **load, "protocol": protocol, "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": None if native else math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label, "config": config}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    crash_events = []
    result["failure_events"] = crash_events
    cancel = threading.Event()
    injector = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}', flush=True)
    try:
        parallel(stop_vm_processes, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol)),
                log=folder / f"node-{vm['index']}-init.log")
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and (native or status[0]['listener']):
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + (0 if native else CLIENT_START_LEAD_SEC)
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['observation_start'] = start_at
        if clients:
            config['scheduled_client_start'] = start_at
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if clients and time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        injection = injector.submit(inject_failures, nodes, trial, start_at, cancel, crash_events)
        planned = {node: start_at + offset for offset, node in FAILURE_EVENTS}
        workload_end = start_at + CLIENT_DURATION_SEC
        while True:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if injection.done():
                injection.result()
            # The injector independently checks that each target was alive at its deadline.
            if any((state['done'] or state.get('error')) and
                   time.time() < planned.get(vm['index'], math.inf)
                   for vm, state in zip(nodes, replica_states)):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if time.time() >= workload_end or (clients and all(s['done'] for s in states)):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        if time.time() < workload_end:
            raise RuntimeError("Clients completed before the configured workload end")
        injection.result()
        if any(event["sent_at"] >= workload_end for event in crash_events):
            raise RuntimeError("Crash injection missed the workload window")
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_started_at'] = min(starts) if starts else None
        result['client_start_skew_sec'] = max(starts) - min(starts) if starts else 0
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'observation_duration_reached'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        cancel.set()
        injector.shutdown(wait=True)
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        result['timings_sec'] = timings
        save_json(logs_folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, load, repetition, index):
    return (f"{protocol}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the Node failure plot cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def main_gcp_failure(work_dir):
    global REMOTE_SESSION, REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if NUM_NODES < 4 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and at least four replicas")
    if 'forget_it' in PROTOCOLS and (NUM_NODES - 1) % 3:
        raise ValueError("Forget-IT requires n=3f+1 (4, 7, 10, ...); 8 is unsupported")
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - SUPPORTED_PROTOCOLS:
        raise ValueError("Choose distinct protocol names from SUPPORTED_PROTOCOLS")
    offsets = [offset for offset, node in FAILURE_EVENTS]
    targets = [node for offset, node in FAILURE_EVENTS]
    if (len(targets) > (NUM_NODES - 1) // 3 or len(set(targets)) != len(targets)
            or any(type(node) is not int or not 2 <= node <= NUM_NODES for node in targets)
            or any(type(sec) is not int or not 0 < sec < CLIENT_DURATION_SEC for sec in offsets)
            or offsets != sorted(set(offsets))):
        raise ValueError("Choose up to f distinct non-ingress replicas at increasing seconds inside the workload")
    if not (0 <= MEASUREMENT_START_SEC < MEASUREMENT_END_SEC <= CLIENT_DURATION_SEC):
        raise ValueError("Invalid measurement window")
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE, ITHS_BLOG_DELTA_MS, FORGET_IT_DELTA_MS) <= 0:
        raise ValueError("Counts, batch size, and Delta must be positive")
    if not re.fullmatch(r'WORKLOAD_(W|R0)', WORKLOAD):
        raise ValueError("This comparison uses write-only transactions; choose WORKLOAD_W or WORKLOAD_R0")
    if not LOAD_POINTS and any(p != 'scp' for p in PROTOCOLS):
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Only remote working paths need unique IDs. Local results use the fixed input.
    session = Path(work_dir)
    REMOTE_SESSION = REMOTE_ROOT + '/' + session_id
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'workload': WORKLOAD, 'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'failure_schedule': FAILURE_EVENTS, 'measurement_start_sec': MEASUREMENT_START_SEC, 'measurement_end_sec': MEASUREMENT_END_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'pbft_timeout_ms': PBFT_TIMEOUT_MS,
              'iths_blog_delta_ms': ITHS_BLOG_DELTA_MS, 'forget_it_delta_ms': FORGET_IT_DELTA_MS,
              'shabdiz_fast_path_only': SHABDIZ_FAST_PATH_ONLY, 'forced_collection': False,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocols': PROTOCOLS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(protocol, repetition, index, load) for protocol in PROTOCOLS
              for repetition in range(1, REPETITIONS+1)
              for index, load in ([(0, {})] if protocol == 'scp' else enumerate(LOAD_POINTS, 1))]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    print(f'IT-HS Delta={ITHS_BLOG_DELTA_MS} ms: its per-batch proposal wait is {2*ITHS_BLOG_DELTA_MS} ms.')
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_SESSION]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before running IT-HS/scheduled clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOLS[0]
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for protocol in PROTOCOLS:
            protocol_trials = [trial for trial in trials if trial[0] == protocol]
            print(f'{protocol}: reusing common binaries; selecting runtime settings.', flush=True)
            for _, repetition, index, load in protocol_trials:
                result = run_trial(vms, builder, art, protocol, load, repetition, index, session, common, client_art)
                results.append(result)
                save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] != 'ok']
    if failed:
        print(f'{len(failed)} trials failed; VMs kept. Set RESUME_SESSION=True to rerun all trials with existing builds.')
    else:
        finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_failure():
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Config archives, setup logs, timing files, and status JSON are temporary.
    # RESULTS_DIR retains the trial status and crash timestamps alongside the client and replica logs.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-failure-') as work_dir:
        return main_gcp_failure(work_dir)


GCP_FAILURE_SESSION = run_gcp_failure()



Results: /home/tejas/work/experiments/gcp-node-failure
1 trials; 0.1 hours of workload, plus builds and setup; pkill at workload end.
IT-HS Delta=1000 ms: its per-batch proposal wait is 2000 ms.
Creating tsm-sc-failure-node-01 in us-central1-c...
Creating tsm-sc-failure-node-02 in us-central1-c...
Creating tsm-sc-failure-node-03 in us-central1-c...
Creating tsm-sc-failure-node-04 in us-central1-c...
Creating tsm-sc-failure-node-05 in us-central1-c...
Creating tsm-sc-failure-node-06 in us-central1-c...
Creating tsm-sc-failure-node-07 in us-central1-c...
Creating tsm-sc-failure-node-08 in us-central1-c...
Creating tsm-sc-failure-node-09 in us-central1-c...
Creating tsm-sc-failure-node-10 in us-central1-c...
Creating tsm-sc-failure-client-01 in us-central1-c...
tsm-sc-failure-node-04: created
tsm-sc-failure-node-03: created
tsm-sc-failure-node-01: created
tsm-sc-failure-node-05: created
tsm-sc-failure-node-06: created
tsm-sc-failure-node-08: created
tsm-sc-failure-node-10: created
tsm-sc-

# Performance vs Read Ratios

In [ ]:
"""GCP performance vs read ratio. Run independently from the repository root.
Each client generates the configured READ/UPDATE mix. READs receive an immediate
ingress ACK. UPDATEs take the selected protocol's consensus path.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time
import uuid

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_DIR = Path.home() / "work/experiments/gcp-read-ratio"
# Fixed output folder. Rerunning the same protocol/size/ratio/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False omits compiler logs.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
# Both modes run ALL configured trials into RESULTS_DIR; no automatic timestamp folder.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-read-ratio"
INSTANCE_PREFIX = "tsm-sc-read-ratio"         # Dedicated read-ratio fleet.
NUM_NODES = 10                         # Forget-IT requires 3f+1: 4, 7, 10, ...
REPLICA_ZONES = ["us-central1-a"] * NUM_NODES
CLIENT_ZONES = ["us-central1-a"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # Incremental make in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOLS = ["shabdiz", "iths", "pbft_fixed", "pbft_rotating", "tetrabft",
             "simple_it", "simple_it_bracha", "forget_it"]
# pbft is an alias of pbft_fixed. Both distinct PBFT and Simple-IT variants run.
SUPPORTED_READ_RATIOS = (0, 10, 20, 25, 40, 50, 60, 75, 80, 100)
READ_RATIOS = [10, 40]                  # Active points from the original experiment.
SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 210
MEASUREMENT_START_SEC = 110
MEASUREMENT_END_SEC = 150
STARTUP_TIMEOUT_SEC = 270
CLIENT_START_LEAD_SEC = 45              # Common wall-clock start across client VMs.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
ITHS_BLOG_DELTA_MS = 1000               # Explicit: 2 Delta wait EACH sequential batch.
PBFT_TIMEOUT_MS = 2000                  # Initial timeout for stalled PBFT work.
FORGET_IT_DELTA_MS = 1000               # Recovery timing; preserves current default.
SHABDIZ_FAST_PATH_ONLY = False          # True: memory-only block archive; keep stellar-core.log.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},  # 300 total
]

COMMIT_TAGS = {"shabdiz": ["[FAST COMMITTED]"],
               "iths": ["[IT-HS COMMITTED]", "[IT-HS-BLOG COMMITTED]"],
               "pbft_fixed": ["[PBFT COMMITTED-LOCAL]"], "pbft_rotating": ["[PBFT COMMITTED-LOCAL]"],
               "tetrabft": ["[TETRA COMMITTED]"],
               "simple_it": ["[SIMPLE-IT COMMITTED]"],
               "simple_it_bracha": ["[SIMPLE-IT COMMITTED]"],
               "forget_it": ["[FORGET-IT COMMITTED]"]}


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol, read_ratio):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': f'WORKLOAD_R{read_ratio}',
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_ITHS_DELTA_MS': str(ITHS_BLOG_DELTA_MS),
        'STELLAR_EXPERIMENT_PBFT_TIMEOUT_MS': str(PBFT_TIMEOUT_MS),
        'STELLAR_EXPERIMENT_FORGET_IT_DELTA_MS': str(FORGET_IT_DELTA_MS),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': str(SHABDIZ_FAST_PATH_ONLY).lower(),
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': '2147483647',
    }


def runtime_command(args, protocol, read_ratio):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol, read_ratio).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOLS[0], READ_RATIOS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-client-load", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOLS[0] / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol, read_ratio):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol, read_ratio)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, read_ratio, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, read_ratio, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    save_json(logs_folder / 'result.json', {'status': 'in_progress'})
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    # Each retry gets a new directory; preserve the failed attempt logs.
    trial = REMOTE_SESSION + "/runs/" + label + "/attempt-" + uuid.uuid4().hex
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "read_ratio_percent": read_ratio,
              "workload": f"WORKLOAD_R{read_ratio}", "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        parallel(stop_vm_processes, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol, read_ratio)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol, read_ratio)),
                log=folder / f"node-{vm['index']}-init.log")
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol, read_ratio),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol, read_ratio),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = start_at + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        # At 100% reads the ingress can ACK every request without committing any blocks.
        if read_ratio < 100 and result['status'] == 'ok':
            def count_commits(vm):
                with (logs_folder / f"node-{vm['index']}" / 'stellar-core.log').open(errors='replace') as log:
                    return sum(line.count(tag) for line in log for tag in COMMIT_TAGS[protocol])
            counts = parallel(count_commits, nodes)
            result['replica_commits'] = counts
            if not all(counts):
                result.update(status='failed', error='A replica logged no commits')
        result['timings_sec'] = timings
        save_json(logs_folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, read_ratio, load, repetition, index):
    return (f"{protocol}/n{NUM_NODES}/read{read_ratio}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the Performance vs Read Ratios cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def main_gcp_read_ratio(work_dir):
    global REMOTE_SESSION, REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if NUM_NODES < 4 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and at least four replicas")
    if 'forget_it' in PROTOCOLS and (NUM_NODES - 1) % 3:
        raise ValueError("Forget-IT requires n=3f+1 (4, 7, 10, ...); 8 is unsupported")
    if not PROTOCOLS or len(set(PROTOCOLS)) != len(PROTOCOLS) or set(PROTOCOLS) - COMMIT_TAGS.keys():
        raise ValueError("Choose distinct custom-client protocols from COMMIT_TAGS; SCP is diagnostic only")
    if not (0 <= MEASUREMENT_START_SEC < MEASUREMENT_END_SEC <= CLIENT_DURATION_SEC):
        raise ValueError("Invalid measurement window")
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE, ITHS_BLOG_DELTA_MS, FORGET_IT_DELTA_MS) <= 0:
        raise ValueError("Counts, batch size, and Delta must be positive")
    if (not READ_RATIOS or any(type(r) is not int or r not in SUPPORTED_READ_RATIOS for r in READ_RATIOS)
            or len(set(READ_RATIOS)) != len(READ_RATIOS)):
        raise ValueError(f'Choose distinct READ_RATIOS from {SUPPORTED_READ_RATIOS}')
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Only remote working paths need unique IDs. Local results use the fixed input.
    session = Path(work_dir)
    REMOTE_SESSION = REMOTE_ROOT + '/' + session_id
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'measurement_start_sec': MEASUREMENT_START_SEC, 'measurement_end_sec': MEASUREMENT_END_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'pbft_timeout_ms': PBFT_TIMEOUT_MS,
              'iths_blog_delta_ms': ITHS_BLOG_DELTA_MS, 'forget_it_delta_ms': FORGET_IT_DELTA_MS,
              'shabdiz_fast_path_only': SHABDIZ_FAST_PATH_ONLY, 'forced_collection': False,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocols': PROTOCOLS, 'read_ratios': READ_RATIOS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(protocol, ratio, repetition, index, load) for protocol in PROTOCOLS for ratio in READ_RATIOS
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    print(f'IT-HS Delta={ITHS_BLOG_DELTA_MS} ms: its per-batch proposal wait is {2*ITHS_BLOG_DELTA_MS} ms.')
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_SESSION]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before running IT-HS/scheduled clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOLS[0]
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for protocol, ratio, repetition, index, load in trials:
            result = run_trial(vms, builder, art, protocol, ratio, load, repetition, index,
                               session, common, client_art)
            results.append(result)
            save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; VMs kept. Set RESUME_SESSION=True to rerun all trials with existing builds.')
    else:
        finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_read_ratio():
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Config archives, setup logs, timing files, and status JSON are temporary.
    # Per-trial status and logs remain available for plotting.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-read-ratio-') as work_dir:
        return main_gcp_read_ratio(work_dir)


GCP_READ_RATIO_SESSION = run_gcp_read_ratio()



# GCP Collection vs Collection Rounds Experiment

In [ ]:
"""GCP Shabdiz collection rounds. Run this cell independently after the local check.
A round is one view in which the designated leader sends COLLECT and a slow proposal.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_ROOT = Path.home() / "work/experiments/gcp-collection-rounds"
# Fixed output folder. Rerunning the same size/round-count/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False omits compiler logs.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-collection-rounds"
INSTANCE_PREFIX = "tsm-sc-collection-rounds"         # Dedicated collection-rounds fleet.
NUM_NODES = 10
REPLICA_ZONES = ["us-central1-c"] * NUM_NODES
CLIENT_ZONES = ["us-central1-c"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # Incremental make in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOL = "shabdiz"
FORCE_COLLECT_AFTER_SEC = 90  # Elapsed from each server process start.
# COLLECTION_ROUNDS = [5, 50, 100, 300]  # Add 0 for a no-forcing control.
COLLECTION_ROUNDS = [50]  # Add 0 for a no-forcing control.

SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 180
STARTUP_TIMEOUT_SEC = 240
CLIENT_START_LEAD_SEC = 0               # Start on readiness; avoid consuming the server interval.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 200 total
]

from importlib import reload
from scripts import collection_rounds

# A notebook kernel can outlive a git pull that changes the measurement schema.
reload(collection_rounds)


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol, rounds):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': 'WORKLOAD_W',
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': 'false',
        'STELLAR_EXPERIMENT_COLLECTION_START_VIEW': '0',
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': str(FORCE_COLLECT_AFTER_SEC),
        'STELLAR_EXPERIMENT_MAX_COLLECT_ATTEMPTS': str(rounds),
    }


def runtime_command(args, protocol, rounds):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol, rounds).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOL, COLLECTION_ROUNDS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-client-load", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol, rounds):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol, rounds)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, rounds, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, rounds, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    save_json(logs_folder / 'result.json', {'status': 'in_progress'})
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    trial = REMOTE_ROOT + "/runs/" + label
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "collection_rounds": rounds, "force_collect_after_sec": FORCE_COLLECT_AFTER_SEC,
              "workload": "WORKLOAD_W", "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        def reset_trial(vm):
            stop_vm_processes(vm)
            ssh(vm, shlex.join(["rm", "-rf", "--", trial]))
        parallel(reset_trial, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol, rounds)
        def prepare_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol, rounds)),
                log=folder / f"node-{vm['index']}-init.log")
        parallel(prepare_node, nodes)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol, rounds),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC if CLIENT_START_LEAD_SEC else 0
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at or None
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol, rounds),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if start_at and time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = (start_at or time.time()) + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_started_at'] = min(starts)
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        result['config'] = config
        if result['status'] == 'ok':
            try:
                result.update(collection_rounds.observed_rounds(logs_folder.glob('node-*/stellar-core.log'), config, result['client_started_at']))
            except (OSError, ValueError) as exc:
                result.update(status='failed', error=str(exc))
        result['timings_sec'] = timings
        save_json(logs_folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, rounds, load, repetition, index):
    return (f"{protocol}/n{NUM_NODES}/after{FORCE_COLLECT_AFTER_SEC}s/rounds{rounds}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the GCP collection rounds plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def main_gcp_collection_rounds(work_dir):
    global REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if PROTOCOL != "shabdiz":
        raise ValueError("Collection trials require Shabdiz")
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if not 4 <= NUM_NODES <= 256 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and 4..256 replicas")
    collection_rounds.check_round_settings(FORCE_COLLECT_AFTER_SEC, COLLECTION_ROUNDS)
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(CLIENT_DURATION_SEC, REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE) <= 0:
        raise ValueError("Duration, counts and batch size must be positive")
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Session bookkeeping is temporary; raw logs use fixed per-trial paths.
    session = Path(work_dir)
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocol': PROTOCOL, 'collection_rounds': COLLECTION_ROUNDS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(rounds, repetition, index, load) for rounds in COLLECTION_ROUNDS
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_ROOT]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before scheduling clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOL
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for rounds, repetition, index, load in trials:
            result = run_trial(vms, builder, art, PROTOCOL, rounds, load, repetition, index,
                               session, common, client_art)
            results.append(result)
            save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; VMs kept. Set RESUME_SESSION=True to rerun all trials with existing builds.')
    else:
        finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_collection_rounds():
    global RESULTS_DIR
    RESULTS_DIR = RESULTS_ROOT.expanduser().resolve()
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Config archives, setup logs, timing files, and status JSON are temporary.
    # Per-trial status and logs remain available for plotting.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-collection-rounds-') as work_dir:
        return main_gcp_collection_rounds(work_dir)


GCP_COLLECTION_ROUNDS_SESSION = run_gcp_collection_rounds()



# Collection vs Network Latency

In [3]:
"""GCP Shabdiz collection versus network latency. Each placement uses half of the replicas in each zone.
A round is one view in which the designated leader sends COLLECT and a slow proposal.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_ROOT = Path.home() / "work/experiments/gcp-collection-network-latency"
# Fixed output folder. Rerunning the same size/round-count/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False omits compiler logs.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-collection-network-latency"
FLEET_PREFIX = "tsm-sc-netlat"
NUM_NODES = 10
BASE_REPLICA_ZONE = "us-west1-b"
REMOTE_REPLICA_ZONES = [
    "europe-west3-c",
    "northamerica-northeast2-b",
    "asia-northeast1-b", "asia-south1-c", "us-west1-b",
]
CLIENT_ZONES = ["us-west1-b"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # Incremental make in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOL = "shabdiz"
FORCE_COLLECT_AFTER_SEC = 100  # Elapsed from each server process start.
# COLLECTION_ROUNDS = [5, 50, 100, 300]  # Add 0 for a no-forcing control.
COLLECTION_ROUNDS = [5]  # Add 0 for a no-forcing control.

SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 210
STARTUP_TIMEOUT_SEC = 240
CLIENT_START_LEAD_SEC = 0               # Start on readiness; avoid consuming the server interval.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 300 total
]

from importlib import reload
from scripts import collection_rounds

# A notebook kernel can outlive a git pull that changes the measurement schema.
reload(collection_rounds)


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol, rounds):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': 'WORKLOAD_W',
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': 'false',
        'STELLAR_EXPERIMENT_COLLECTION_START_VIEW': '0',
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': str(FORCE_COLLECT_AFTER_SEC),
        'STELLAR_EXPERIMENT_MAX_COLLECT_ATTEMPTS': str(rounds),
    }


def runtime_command(args, protocol, rounds):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol, rounds).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOL, COLLECTION_ROUNDS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-collection-latency", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol, rounds):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol, rounds)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, rounds, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, rounds, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    save_json(logs_folder / 'result.json', {'status': 'in_progress'})
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    trial = REMOTE_ROOT + "/runs/" + label
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "collection_rounds": rounds, "force_collect_after_sec": FORCE_COLLECT_AFTER_SEC,
              "workload": "WORKLOAD_W", "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        def reset_trial(vm):
            stop_vm_processes(vm)
            ssh(vm, shlex.join(["rm", "-rf", "--", trial]))
        parallel(reset_trial, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol, rounds)
        def prepare_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol, rounds)),
                log=folder / f"node-{vm['index']}-init.log")
        parallel(prepare_node, nodes)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol, rounds),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC if CLIENT_START_LEAD_SEC else 0
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at or None
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol, rounds),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if start_at and time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = (start_at or time.time()) + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_started_at'] = min(starts)
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        result['config'] = config
        if result['status'] == 'ok':
            try:
                result.update(collection_rounds.observed_rounds(logs_folder.glob('node-*/stellar-core.log'), config, result['client_started_at']))
            except (OSError, ValueError) as exc:
                result.update(status='failed', error=str(exc))
        result['timings_sec'] = timings
        save_json(logs_folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, rounds, load, repetition, index):
    return (f"{protocol}/n{NUM_NODES}/after{FORCE_COLLECT_AFTER_SEC}s/rounds{rounds}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the Collection vs Network Latency plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def run_latency_placement(work_dir, remote_zone):
    global REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if PROTOCOL != "shabdiz":
        raise ValueError("Collection trials require Shabdiz")
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if not 4 <= NUM_NODES <= 256 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and 4..256 replicas")
    collection_rounds.check_round_settings(FORCE_COLLECT_AFTER_SEC, COLLECTION_ROUNDS)
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(CLIENT_DURATION_SEC, REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE) <= 0:
        raise ValueError("Duration, counts and batch size must be positive")
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Session bookkeeping is temporary; raw logs use fixed per-trial paths.
    session = Path(work_dir)
    REMOTE_BUILD = IMAGE_REPO
    common = {'base_replica_zone': BASE_REPLICA_ZONE, 'remote_replica_zone': remote_zone,
              'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocol': PROTOCOL, 'collection_rounds': COLLECTION_ROUNDS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(rounds, repetition, index, load) for rounds in COLLECTION_ROUNDS
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_ROOT]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before scheduling clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOL
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for rounds, repetition, index, load in trials:
            result = run_trial(vms, builder, art, PROTOCOL, rounds, load, repetition, index,
                               session, common, client_art)
            results.append(result)
            save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; VMs kept. Set RESUME_SESSION=True to rerun all trials with existing builds.')
    else:
        finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_network_latency():
    global REPLICA_ZONES, INSTANCE_PREFIX, RESULTS_DIR
    if NUM_NODES % 2:
        raise ValueError("Use an even replica count for an equal split between zones")
    if not REMOTE_REPLICA_ZONES or len(set(REMOTE_REPLICA_ZONES)) != len(REMOTE_REPLICA_ZONES):
        raise ValueError("Specify distinct remote replica zones")
    zones = [BASE_REPLICA_ZONE, *REMOTE_REPLICA_ZONES, *CLIENT_ZONES]
    if any(not re.fullmatch(r"[a-z][a-z0-9-]*[0-9]-[a-z]", zone) for zone in zones):
        raise ValueError(f"Invalid GCP zone in {zones}")
    root = RESULTS_ROOT.expanduser().resolve()
    for remote_zone in REMOTE_REPLICA_ZONES:
        REPLICA_ZONES = [BASE_REPLICA_ZONE] * (NUM_NODES // 2) + [remote_zone] * (NUM_NODES // 2)
        INSTANCE_PREFIX = f"{FLEET_PREFIX}-{remote_zone}"
        RESULTS_DIR = root / f"base-{BASE_REPLICA_ZONE}" / f"remote-{remote_zone}"
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Replica placement: {NUM_NODES//2} in {BASE_REPLICA_ZONE}, {NUM_NODES//2} in {remote_zone}")
        with tempfile.TemporaryDirectory(prefix="stellar-netlat-") as work_dir:
            run_latency_placement(work_dir, remote_zone)
    return root


GCP_NETWORK_LATENCY_RESULTS = run_gcp_network_latency()


Replica placement: 5 in us-west1-b, 5 in europe-west3-c
Results: /home/tejas/work/experiments/gcp-collection-network-latency/base-us-west1-b/remote-europe-west3-c
1 trials; 0.1 hours of workload, plus builds and setup; pkill at workload end.
Creating tsm-sc-netlat-europe-west3-c-node-01 in us-west1-b...
Creating tsm-sc-netlat-europe-west3-c-node-02 in us-west1-b...
Creating tsm-sc-netlat-europe-west3-c-node-03 in us-west1-b...
Creating tsm-sc-netlat-europe-west3-c-node-04 in us-west1-b...
Creating tsm-sc-netlat-europe-west3-c-node-05 in us-west1-b...
Creating tsm-sc-netlat-europe-west3-c-node-06 in europe-west3-c...
Creating tsm-sc-netlat-europe-west3-c-node-07 in europe-west3-c...
Creating tsm-sc-netlat-europe-west3-c-node-08 in europe-west3-c...
Creating tsm-sc-netlat-europe-west3-c-node-09 in europe-west3-c...
Creating tsm-sc-netlat-europe-west3-c-node-10 in europe-west3-c...
Creating tsm-sc-netlat-europe-west3-c-client-01 in us-west1-b...
tsm-sc-netlat-europe-west3-c-node-01: creat

RuntimeError: Command failed (1): gcloud compute instances
WARNING: Some requests generated warnings:
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert

ERROR: (gcloud.compute.instances.create) Could not fetch resource:
 - Quota 'SSD_TOTAL_GB' exceeded.  Limit: 500.0 in region us-west1.
	metric name = compute.googleapis.com/ssd_total_storage
	limit name = SSD-TOTAL-GB-per-project-region
	limit = 500.0
	dimensions = region: us-west1
Try your request in another zone, or view documentation on how to increase quotas: https://cloud.google.com/compute/quotas.

; Command failed (1): gcloud compute instances
WARNING: Some requests generated warnings:
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the r

# GCP Collection vs System Size Experiment

In [4]:
"""GCP Shabdiz collection versus system size. Run this cell independently.
Build the largest fleet once, then delete surplus replicas before each smaller size.
A round is one view in which the designated leader sends COLLECT and a slow proposal.
"""
import concurrent.futures
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_ROOT = Path.home() / "work/experiments/gcp-collection-system-size"
# Fixed output folder. Rerunning the same size/round-count/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False omits compiler logs.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-collection-system-size"
INSTANCE_PREFIX = "tsm-sc-collect-size"         # Dedicated system-size collection fleet.
SYSTEM_SIZES = [4, 8, 16, 24]           # Replica counts; always run largest first.
REPLICA_ZONE = "us-central1-c"
CLIENT_ZONES = ["us-central1-c"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True also deletes the final fleet after a successful sweep.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # Incremental make in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOL = "shabdiz"
FORCE_COLLECT_AFTER_SEC = 90  # Elapsed from each server process start.
COLLECTION_ROUNDS = 50                  # Same forced round count at every system size.

SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 180
STARTUP_TIMEOUT_SEC = 240
CLIENT_START_LEAD_SEC = 0               # Start on readiness; avoid consuming the server interval.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},  # 200 total
]

from importlib import reload
from scripts import collection_rounds

# A notebook kernel can outlive a git pull that changes the measurement schema.
reload(collection_rounds)


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol, rounds):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': 'WORKLOAD_W',
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': 'false',
        'STELLAR_EXPERIMENT_COLLECTION_START_VIEW': '0',
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': str(FORCE_COLLECT_AFTER_SEC),
        'STELLAR_EXPERIMENT_MAX_COLLECT_ATTEMPTS': str(rounds),
    }


def runtime_command(args, protocol, rounds):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol, rounds).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOL, COLLECTION_ROUNDS)):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session, max_nodes):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", [REPLICA_ZONE] * max_nodes), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-collection-size", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(max_nodes)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            # The dedicated rule may have been created for a smaller sweep.
            ports = sorted(allowed.union(expected_ports), key=lambda p: int(p.split("-")[0]))
            cloud("firewall-rules", "update", rule,
                  "--allow", ",".join("tcp:" + p for p in ports))
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, expected_peers=None):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {expected_peers is not None!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {expected_peers!r} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol, rounds):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {len(node_ips)}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({len(node_ips)}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{len(node_ips)})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol, rounds)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, rounds, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, common["num_nodes"], rounds, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    save_json(logs_folder / 'result.json', {'status': 'in_progress'})
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    trial = REMOTE_ROOT + "/runs/" + label
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "collection_rounds": rounds, "force_collect_after_sec": FORCE_COLLECT_AFTER_SEC,
              "workload": "WORKLOAD_W", "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        def reset_trial(vm):
            stop_vm_processes(vm)
            ssh(vm, shlex.join(["rm", "-rf", "--", trial]))
        parallel(reset_trial, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol, rounds)
        def prepare_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol, rounds)),
                log=folder / f"node-{vm['index']}-init.log")
        parallel(prepare_node, nodes)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol, rounds),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", expected_peers=len(nodes) - 1), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC if CLIENT_START_LEAD_SEC else 0
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at or None
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol, rounds),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if start_at and time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = (start_at or time.time()) + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_started_at'] = min(starts)
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        result['config'] = config
        if result['status'] == 'ok':
            try:
                result.update(collection_rounds.observed_rounds(logs_folder.glob('node-*/stellar-core.log'), config, result['client_started_at']))
            except (OSError, ValueError) as exc:
                result.update(status='failed', error=str(exc))
        result['timings_sec'] = timings
        save_json(logs_folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def trial_label(protocol, num_nodes, rounds, load, repetition, index):
    return (f"{protocol}/n{num_nodes}/after{FORCE_COLLECT_AFTER_SEC}s/rounds{rounds}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def delete_vms(vms):
    print('Deleting:', ', '.join(vm['name'] for vm in vms), flush=True)
    parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900), vms)


def finish_session(vms):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the GCP collection vs system size plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [vm for vm in vms if (vm['name'], vm['zone']) in existing]
        if remaining:
            delete_vms(remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def main_gcp_collection_system_size(work_dir):
    global REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if PROTOCOL != "shabdiz":
        raise ValueError("Collection trials require Shabdiz")
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if (not SYSTEM_SIZES or any(type(n) is not int or not 4 <= n <= 256 for n in SYSTEM_SIZES)
            or len(set(SYSTEM_SIZES)) != len(SYSTEM_SIZES)):
        raise ValueError("SYSTEM_SIZES must contain distinct integer replica counts from 4 to 256")
    sizes = sorted(SYSTEM_SIZES, reverse=True)
    collection_rounds.check_round_settings(FORCE_COLLECT_AFTER_SEC, [COLLECTION_ROUNDS])
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(CLIENT_DURATION_SEC, REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE) <= 0:
        raise ValueError("Duration, counts and batch size must be positive")
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Session bookkeeping is temporary; raw logs use fixed per-trial paths.
    session = Path(work_dir)
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha,
              'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'system_sizes': sizes, 'replica_zone': REPLICA_ZONE, 'protocol': PROTOCOL, 'collection_rounds': COLLECTION_ROUNDS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(size, repetition, index, load) for size in sizes
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    vms = fleet(session, sizes[0])
    parallel(wait_ssh, vms)
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_ROOT]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before scheduling clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOL
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for size, repetition, index, load in trials:
            surplus = [vm for vm in vms if vm['role'] == 'node' and vm['index'] > size]
            if surplus:
                # Previous trials have already stopped their processes and downloaded logs.
                # Keep final process cleanup scoped to the retained fleet even if deletion fails.
                vms = [vm for vm in vms if vm['role'] == 'client' or vm['index'] <= size]
                delete_vms(surplus)
            trial_config = {**common, 'num_nodes': size, 'replica_zones': [REPLICA_ZONE] * size}
            result = run_trial(vms, builder, art, PROTOCOL, COLLECTION_ROUNDS, load,
                               repetition, index, session, trial_config, client_art)
            results.append(result)
            save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; the remaining VMs were kept. See the per-trial results for details.')
    else:
        finish_session(vms)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_collection_system_size():
    global RESULTS_DIR
    RESULTS_DIR = RESULTS_ROOT.expanduser().resolve()
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Config archives, setup logs, timing files, and status JSON are temporary.
    # Per-trial status and logs remain available for plotting.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-collection-system-size-') as work_dir:
        return main_gcp_collection_system_size(work_dir)


GCP_COLLECTION_SYSTEM_SIZE_RESULTS = run_gcp_collection_system_size()

Results: /home/tejas/work/experiments/gcp-collection-system-size
4 trials; 0.2 hours of workload, plus builds and setup; pkill at workload end.
Creating tsm-sc-collect-size-node-01 in us-central1-c...Creating tsm-sc-collect-size-node-02 in us-central1-c...

Creating tsm-sc-collect-size-node-03 in us-central1-c...
Creating tsm-sc-collect-size-node-04 in us-central1-c...
Creating tsm-sc-collect-size-node-05 in us-central1-c...
Creating tsm-sc-collect-size-node-06 in us-central1-c...
Creating tsm-sc-collect-size-node-07 in us-central1-c...
Creating tsm-sc-collect-size-node-08 in us-central1-c...
Creating tsm-sc-collect-size-node-09 in us-central1-c...
Creating tsm-sc-collect-size-node-10 in us-central1-c...
Creating tsm-sc-collect-size-node-11 in us-central1-c...
Creating tsm-sc-collect-size-node-12 in us-central1-c...
Creating tsm-sc-collect-size-node-13 in us-central1-c...
Creating tsm-sc-collect-size-node-14 in us-central1-c...
Creating tsm-sc-collect-size-node-15 in us-central1-c...
C


KeyboardInterrupt



In [ ]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")

In [ ]:
#             # Clean remote stellar-private.
#             results = run_parallel(
#                 clean_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Copy fresh stellar-private config to all active machines.
#             results = run_parallel(
#                 copy_folder_to_instance,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Kill old processes on all active machines.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Start consensus replicas.
#             results = run_parallel(
#                 run_stellar_private,
#                 range(num_nodes),
#                 max_workers=min(48, num_nodes),
#             )

#             print(results)
#             print("All Stellar nodes should be starting in the background.")

#             # Give nodes time to authenticate and start the client listener.
#             time.sleep(60)

#             # Start only required client VMs for this load point.
#             active_client_indices = [
#                 num_nodes + j for j in range(active_clients)
#             ]

#             results = run_parallel(
#                 lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print(results)
#             print(
#                 f"Started {active_clients} client VM(s), "
#                 f"each with {client_threads} client threads. "
#                 f"Total client threads = {total_client_threads}. "
#                 f"Aggregate max in-flight = {aggregate_max_in_flight}."
#             )

#             # Let the system run before forcing failures.
#             time.sleep(CLIENT_WAIT_AFTER_START_SEC)


#             # Stop all nodes and clients.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # -----------------------------------------------------------------
#             # Save logs.
#             # -----------------------------------------------------------------
#             remote_base_folder = "/home/tejas/stellar-private"

#             local_base_destination = (
#                 "/home/tejas/work/experiments/shabdiz/"
#                 + f"Collection_{max_collect_attempts}_"
#                 + f"nodes_{num_nodes}_{run_label}"
#             )

#             Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#             # Save run metadata.
#             with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#                 f.write(f"num_nodes={num_nodes}\n")
#                 f.write(f"max_collect_attempts={max_collect_attempts}\n")
#                 f.write(f"force_collect_after_sec=100\n")
#                 f.write(f"active_clients={active_clients}\n")
#                 f.write(f"client_threads_per_vm={client_threads}\n")
#                 f.write(f"total_client_threads={total_client_threads}\n")
#                 f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#                 f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#                 f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#                 f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#                 f.write(f"leader_ip={node1_ip}\n")
#                 f.write(f"machine_type={machine_type}\n")
#                 f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#             def copy_folder_from_instance(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"

#                 node_number = i + 1
#                 node_folder = f"node{node_number}"

#                 remote_source_path = posixpath.join(remote_base_folder, node_folder)

#                 local_destination_path = Path(local_base_destination) / instance_name
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 remote_source = f"{instance_name}:{remote_source_path}"

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#                 print(
#                     f"Executing command to copy {node_folder} "
#                     f"from {instance_name}: {command}"
#                 )

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy from {instance_name} finished with exit code: {output}")

#                 return (instance_name, output)

#             # Copy all replica logs.
#             node_copy_results = run_parallel(
#                 copy_folder_from_instance,
#                 range(3),
#                 max_workers=min(48, max(1, num_nodes)),
#             )

#             def copy_client_log(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"
#                 client_id = i - num_nodes

#                 remote_source = (
#                     f"{instance_name}:/home/tejas/stellar-private/"
#                     f"stellar-client-{client_id}.log"
#                 )

#                 local_destination_path = Path(local_base_destination)
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#                 print(f"Copying client log from {instance_name}...")

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy finished with exit code: {output}")

#                 return (instance_name, output)

#             client_copy_results = run_parallel(
#                 copy_client_log,
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print("\n--- Summary of Download Results ---")
#             print("Node log copies:", node_copy_results)
#             print("Client log copies:", client_copy_results)
#             print(f"Saved run to: {local_base_destination}")

#  Memory Exp

In [ ]:
"""GCP Shabdiz pruning experiment. Compare retained prepare ballots and process RSS on one replica."""
import concurrent.futures
import csv
from datetime import datetime, timezone
import json
import math
import os
from pathlib import Path
import re
import shlex
import shutil
import signal
import socket
import subprocess
import tarfile
import tempfile
import threading
import time

# ---------------- Settings: edit these ----------------
REPO = Path.cwd()                       # Your local updated stellar-core checkout.
RESULTS_ROOT = Path.home() / "work/experiments/gcp-shabdiz-memory"
# Fixed output folder. Rerunning the same scenario/replica/repetition/load replaces its logs.
KEEP_BUILD_LOGS = True                 # False omits compiler logs.
RESUME_SESSION = False                 # True: reuse VM source; check binaries once, not per protocol.
PROJECT = "research-488322"
IMAGE_PROJECT = PROJECT
IMAGE_FAMILY = "tsm-sc-family"          # Same custom image family.
GCP_USER = "tejas"
IMAGE_REPO = "/home/tejas/stellar-core"  # Checkout/toolchain already in the image.
REMOTE_ROOT = "/home/tejas/stellar-gcp-shabdiz-memory"
INSTANCE_PREFIX = "tsm-sc-memory"         # Dedicated memory experiment fleet.
NUM_NODES = 4
MEASURED_REPLICA = 2
REPLICA_ZONES = ["us-central1-a"] * NUM_NODES
CLIENT_ZONES = ["us-central1-a"]
MACHINE_TYPE = "e2-standard-2"
CLIENT_MACHINE_TYPE = "e2-standard-2"
NETWORK = "default"
BOOT_DISK_GB = 50                       # Existing image checkout and experiment logs.
CREATE_MISSING_VMS = True              # Reuse matching dedicated VMs if present.
DELETE_ALL_AFTER_FINAL_RUN = True     # True deletes all replica/client VMs in this run after collecting results.
USE_IAP = False                        # True if direct SSH is unavailable.
BUILD_JOBS = 2                         # Incremental make in the existing checkout on each replica VM.
MAX_PARALLEL_WORKERS = 48               # Parallel VM operations, as in the original runner.
STREAM_BUILD_OUTPUT = False            # Full compiler logs stay on disk; True also prints every line in Jupyter.
SOURCE_COMMIT = None                   # None pins local HEAD (must be pushed).

PROTOCOL = "shabdiz"
SCENARIOS = ["never_pruned", "normal_pruning"]  # Both run without injected failures.

SERVER_BATCH_SIZE = 100
CLIENT_PORT = 12000
CLIENT_DURATION_SEC = 360
STARTUP_TIMEOUT_SEC = 240
CLIENT_START_LEAD_SEC = 0               # Start on readiness; avoid consuming the server interval.
MAX_CLIENT_START_SKEW_SEC = 1.0
SEND_INTERVAL_US = 0
REPETITIONS = 1                        # Use >=3 for final paper measurements.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},  # 300 total
]

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def command(args, *, log=None, timeout=600, stream_label=None):
    args = [str(x) for x in args]
    if log is None:
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, timeout=timeout)
        if result.returncode:
            raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(args[:3])}\n"
                               + result.stderr[-4000:] + result.stdout[-1000:])
        return result.stdout
    with Path(log).open("a") as out:
        out.write("\n$ " + shlex.join(args) + "\n")
        out.flush()
        if stream_label is None or not STREAM_BUILD_OUTPUT:
            # Write directly to disk: no compiler-output accumulation in Jupyter or Python.
            subprocess.run(args, check=True, stdout=out, stderr=subprocess.STDOUT, timeout=timeout)
            return
        # Optional verbose debugging only; full compiler logs are always kept on disk.
        proc = subprocess.Popen(args, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, bufsize=1, start_new_session=True)
        expired = threading.Event()
        def expire():
            if proc.poll() is None:
                expired.set()
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        timer = threading.Timer(timeout, expire)
        timer.daemon = True
        timer.start()
        try:
            for line in proc.stdout:
                out.write(line)
                out.flush()
                print(f'[{stream_label}] {line.rstrip()}', flush=True)
            code = proc.wait()
            if expired.is_set():
                raise subprocess.TimeoutExpired(args, timeout)
            if code:
                raise subprocess.CalledProcessError(code, args)
        finally:
            timer.cancel()
            if proc.poll() is None:
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            proc.wait()
            proc.stdout.close()


def parallel(fn, items):
    # Consume every future: surface all failures after the other workers finish.
    items = list(items)
    if not items:
        return []
    with concurrent.futures.ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_WORKERS, len(items))) as pool:
        futures = [pool.submit(fn, item) for item in items]
        results, errors = [], []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as exc:
                errors.append(exc)
        if errors:
            raise RuntimeError("; ".join(str(e) for e in errors)) from errors[0]
        return results


def cloud(*args, **kwargs):
    return command(["gcloud", "compute", *args, "--project", PROJECT, "--quiet"], **kwargs)


def ssh(vm, script, *, log=None, timeout=600, stream_label=None):
    return cloud("ssh", f"{GCP_USER}@{vm['name']}", "--zone", vm["zone"],
                 *(["--tunnel-through-iap"] if USE_IAP else []),
                 "--command", "bash -lc " + shlex.quote("set -euo pipefail\n" + script),
                 log=log, timeout=timeout, stream_label=stream_label)


def scp(vm, source, destination, *, download=False):
    endpoint = f"{GCP_USER}@{vm['name']}:"
    return cloud("scp", endpoint + str(source) if download else str(source),
                 str(destination) if download else endpoint + str(destination),
                 "--zone", vm["zone"], *(["--tunnel-through-iap"] if USE_IAP else []),
                 timeout=1800)


def python_remote(vm, code, *, log=None, timeout=600):
    return ssh(vm, "python3 -c " + shlex.quote(code), log=log, timeout=timeout)


def runtime_environment(protocol, scenario):
    # Same settings as the former header edits, now selected at process startup.
    return {
        'STELLAR_EXPERIMENT_PROTOCOL': protocol,
        'STELLAR_EXPERIMENT_NONPRUNE': 'true' if scenario == 'never_pruned' else 'false',
        'STELLAR_EXPERIMENT_WORKLOAD': 'WORKLOAD_W',
        'STELLAR_EXPERIMENT_BATCH_SIZE': str(SERVER_BATCH_SIZE),
        'STELLAR_EXPERIMENT_CLIENT_PORT': str(CLIENT_PORT),
        'STELLAR_EXPERIMENT_SHABDIZ_FAST_PATH_ONLY': 'false',
        'STELLAR_EXPERIMENT_COLLECTION_START_VIEW': '0',
        'STELLAR_EXPERIMENT_FORCE_COLLECT_AFTER_SEC': '2147483647',
        'STELLAR_EXPERIMENT_MAX_COLLECT_ATTEMPTS': '0',
    }


def runtime_command(args, protocol, scenario):
    return ['env', *(key + '=' + value for key, value in runtime_environment(protocol, scenario).items()), *args]


def require_runtime_config(header):
    if any(key not in header for key in runtime_environment(PROTOCOL, SCENARIOS[0])):
        raise ValueError('Source lacks runtime experiment settings. Pull the updated source and '
                         'run once with RESUME_SESSION=False (and an updated SOURCE_COMMIT if set).')


def fleet(session):
    existing = json.loads(cloud("instances", "list", "--format=json"))
    if (session / "fleet.json").exists():
        # A family may advance between attempts; resume with the original image.
        image = {"selfLink": json.loads((session / "fleet.json").read_text())["image"]}
    else:
        image = json.loads(command(["gcloud", "compute", "images", "describe-from-family", IMAGE_FAMILY,
                                    "--format=json", "--project", IMAGE_PROJECT]))
    # Validate all existing VM identities before starting any cloud mutations.
    # Each worker then creates/starts and inspects one VM; ordering remains
    # node-01 ... node-N, client-01 ... so the ingress leader never changes.
    requested = []
    for role, zones in [("node", REPLICA_ZONES), ("client", CLIENT_ZONES)]:
        for index, zone in enumerate(zones, 1):
            name = f"{INSTANCE_PREFIX}-{role}-{index:02d}"
            machine = MACHINE_TYPE if role == "node" else CLIENT_MACHINE_TYPE
            matches = [v for v in existing if v["name"] == name]
            if len(matches) > 1:
                raise RuntimeError(f"Ambiguous existing VM name: {name}")
            if matches:
                old = matches[0]
                if old["zone"].split("/")[-1] != zone or old["machineType"].split("/")[-1] != machine:
                    raise RuntimeError(f"Existing {name} has a different zone/machine type; change the prefix")
                if old["status"] not in ("TERMINATED", "RUNNING"):
                    raise RuntimeError(f"{name} is {old['status']}")
            elif not CREATE_MISSING_VMS:
                raise RuntimeError(f"Missing {name}; enable CREATE_MISSING_VMS or correct the prefix")
            requested.append({"name": name, "zone": zone, "role": role, "index": index,
                              "machine_type": machine, "existing": matches[0] if matches else None})

    def provision_vm(spec):
        name, zone, machine = spec['name'], spec['zone'], spec['machine_type']
        old = spec['existing']
        if old is None:
            print(f'Creating {name} in {zone}...', flush=True)
            cloud("instances", "create", name, "--zone", zone, "--machine-type", machine,
                  "--image", image["selfLink"], "--network", NETWORK,
                  "--boot-disk-size", str(BOOT_DISK_GB) + "GB", "--boot-disk-type", "pd-balanced",
                  "--tags", INSTANCE_PREFIX, "--labels", "experiment=stellar-memory", timeout=900)
        elif old['status'] == 'TERMINATED':
            cloud("instances", "start", name, "--zone", zone, timeout=900)
        vm = json.loads(cloud("instances", "describe", name, "--zone", zone, "--format=json"))
        if vm["networkInterfaces"][0]["network"].split("/")[-1] != NETWORK:
            raise RuntimeError(f"{name} is on a different network; change INSTANCE_PREFIX")
        disk_url = vm["disks"][0]["source"]
        disk = json.loads(cloud("disks", "describe", disk_url.split("/")[-1],
                                "--zone", zone, "--format=json"))
        if disk.get("sourceImage") != image["selfLink"]:
            raise RuntimeError(f"{name} does not use the current family image; choose a fresh INSTANCE_PREFIX")
        if INSTANCE_PREFIX not in vm.get("tags", {}).get("items", []):
            cloud("instances", "add-tags", name, "--zone", zone, "--tags", INSTANCE_PREFIX)
        print(f"{name}: {'created' if old is None else 'reused'}", flush=True)
        return {"name": name, "zone": zone, "role": spec['role'], "index": spec['index'],
                "ip": vm["networkInterfaces"][0]["networkIP"],
                "machine_type": machine, "disk": disk_url, "source_image": disk["sourceImage"]}

    vms = parallel(provision_vm, requested)
    # One coordinator writes the manifest after all workers finish.
    save_json(session / "fleet.json", {"image": image["selfLink"], "vms": vms})
    # Internal TCP only, between this fleet's tags (no public client/peer ports).
    rule = INSTANCE_PREFIX + "-internal"
    rules = json.loads(cloud("firewall-rules", "list", "--format=json"))
    expected_ports = [str(CLIENT_PORT)] + [str(11625 + 10*i) for i in range(NUM_NODES)]
    existing_rule = next((r for r in rules if r["name"] == rule), None)
    if existing_rule is None:
        cloud("firewall-rules", "create", rule, "--network", NETWORK, "--direction", "INGRESS",
              "--source-tags", INSTANCE_PREFIX, "--target-tags", INSTANCE_PREFIX,
              "--allow", ",".join("tcp:" + p for p in expected_ports))
    else:
        if (existing_rule.get('disabled', False) or existing_rule.get('direction') != 'INGRESS'
                or existing_rule['network'].split('/')[-1] != NETWORK
                or existing_rule.get('sourceTags') != [INSTANCE_PREFIX]
                or existing_rule.get('targetTags') != [INSTANCE_PREFIX]):
            raise RuntimeError(f"Existing firewall {rule} differs; use another INSTANCE_PREFIX")
        allowed = {p for a in existing_rule.get('allowed', []) if a['IPProtocol'] == 'tcp'
                   for p in a.get('ports', [])}
        if not set(expected_ports) <= allowed:
            raise RuntimeError(f"Firewall {rule} lacks required ports; use another INSTANCE_PREFIX")
    return vms


def wait_ssh(vm):
    end = time.monotonic() + 300
    while True:
        try:
            ssh(vm, "true", timeout=30)
            return
        except Exception:
            if time.monotonic() >= end:
                raise
            time.sleep(5)


def launch_process(vm, root, args, lib, log, start_at=0):
    # Exec the binary so the recorded PID belongs to the actual process.
    command_line = shlex.join(args)
    log_path = log if log.startswith("/") else root + "/" + log
    lines = ['cd ' + shlex.quote(root),
             'export LD_LIBRARY_PATH=' + shlex.quote(lib) + ':"${LD_LIBRARY_PATH:-}"']
    if start_at:
        lines.append('python3 -c ' + shlex.quote(
            f'import time; time.sleep(max(0, {start_at!r} - time.time()))'))
    lines += ['date +%s.%N > started_at', 'exec ' + command_line]
    script = '\n'.join(lines) + '\n'
    code = f"from pathlib import Path; p=Path({root!r}); p.mkdir(parents=True, exist_ok=True); (p/'launch.sh').write_text({script!r})"
    ssh(vm, 'python3 -c ' + shlex.quote(code) + '\n' +
        'cd ' + shlex.quote(root) + '\nnohup bash ' + shlex.quote(root + '/launch.sh') +
        ' > ' + shlex.quote(log_path) + ' 2>&1 < /dev/null &\n' +
        'echo "$!" > ' + shlex.quote(root + '/pid') + '\ndisown')


def process_state(vm, root, *, check_ready=False):
    code = rf'''
import json, pathlib
root = pathlib.Path({root!r})
status = {{'done': False, 'ready': False, 'listener': False, 'returncode': None}}
pid = int((root/'pid').read_text())
try:
    process = pathlib.Path('/proc')/str(pid)
    alive = process.joinpath('stat').read_text().rsplit(')', 1)[1].split()[0] != 'Z'
    alive = alive and process.joinpath('cwd').resolve() == root.resolve()
except FileNotFoundError:
    alive = False
if not alive:
    status['done'] = True
    client_log = root/'client.log'
    if client_log.exists() and '=== Multi-thread Client Summary ===' in client_log.read_text(errors='replace'):
        status['returncode'] = 0
    else:
        status['error'] = 'Process exited before workload ended; inspect console/client log'
if (root/'started_at').exists():
    status['started_at'] = float((root/'started_at').read_text())
# Read-only readiness: authenticate the exact configured identities, not
# just a peer count or a log message. Works when replica logging is disabled.
if {check_ready!r} and alive:
    import re, urllib.request
    try:
        cfg = (root/'stellar-core.cfg').read_text()
        expected = set(re.findall(r'^PUBLIC_KEY\s*=\s*"([^" ]+)"', cfg, re.M))
        port = int(re.search(r'^HTTP_PORT\s*=\s*(\d+)', cfg, re.M)[1])
        with urllib.request.urlopen(f'http://127.0.0.1:{{port}}/peers?fullkeys=true', timeout=5) as response:
            peers = json.load(response).get('authenticated_peers') or {{}}
        connected = {{p['id'] for direction in ('inbound', 'outbound')
                     for p in (peers.get(direction) or [])}}
        status['expected_peers'] = len(expected)
        status['connected_peers'] = len(expected & connected)
        status['missing_peers'] = sorted(expected - connected)
        status['ready'] = len(expected) == {NUM_NODES - 1} and expected <= connected
        # Confirm this process owns the client LISTEN socket without opening
        # a probe connection that would be mistaken for a workload client.
        sockets = set()
        for fd in process.joinpath('fd').iterdir():
            try:
                sockets.add(fd.readlink().as_posix())
            except FileNotFoundError:
                pass
        for table in ('tcp', 'tcp6'):
            net = process/'net'/table
            if net.exists():
                for row in net.read_text().splitlines()[1:]:
                    fields = row.split()
                    if (fields[3] == '0A' and int(fields[1].rsplit(':', 1)[1], 16) == {CLIENT_PORT}
                            and 'socket:[' + fields[9] + ']' in sockets):
                        status['listener'] = True
    except Exception as exc:
        status['probe_error'] = str(exc)
print(json.dumps(status))
'''
    return json.loads(python_remote(vm, code))

REMOTE_BUILD_HELPERS = r'''
import json, pathlib, subprocess, time
timings = {}
def timed(name, args, **kwargs):
    start = time.monotonic()
    print(name + ": starting", flush=True)
    try:
        subprocess.run(args, check=True, **kwargs)
    finally:
        timings[name] = round(time.monotonic() - start, 3)
        print(name + ": " + str(timings[name]) + " s", flush=True)
'''


def build_vm(vm, code, folder, saved_log=None):
    folder.mkdir(parents=True, exist_ok=True)
    log = folder / 'build.log'
    script = 'cd ' + shlex.quote(REMOTE_BUILD) + '\npython3 -c ' + shlex.quote(code)
    start = time.monotonic()
    try:
        ssh(vm, script, log=log, timeout=14400, stream_label=vm['name'])
    finally:
        if KEEP_BUILD_LOGS and saved_log is not None and log.is_file():
            saved_log.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(log, saved_log)
    result = {}
    with log.open(errors='replace') as output:
        for line in output:
            if line.startswith('BUILD_TIMINGS_JSON='):
                result = json.loads(line.partition('=')[2])
    result['wall_sec'] = round(time.monotonic() - start, 3)
    save_json(folder / 'build-timings.json', result)
    print(vm['name'] + ': ' + ('reused build' if result.get('reused') else
          ', '.join(f'{key}={value:.1f}s' for key, value in result.items()
                    if key.endswith('_sec'))), flush=True)


def build_replica(vm, header, folder):
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
# Restore old notebook-generated flags once; never rewrite for protocol/size changes.
if header.read_text() != canonical:
    header.write_text(canonical)
# Build once in the image's ORIGINAL checkout; unchanged inputs make this a no-op.
timed('make_sec', ['make', '-j{BUILD_JOBS}', '-o', 'Makefile', '-o', 'config.status',
                   'MAKE=make -o Makefile -o config.status'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-build.log'))


def build_client(vm, header, folder):
    # One workload-configurable client; compile only when source inputs changed.
    code = REMOTE_BUILD_HELPERS + f"""
header = pathlib.Path('src/overlay/experiments/ExperimentConfig.h')
canonical = {header!r}
if header.read_text() != canonical:
    header.write_text(canonical)
binary = pathlib.Path('shab_client')
inputs = [pathlib.Path(name) for name in ('shab_client.cpp', 'src/overlay/CustomYCSBWorkload.h',
          'src/overlay/CustomProtocolTypes.h', 'src/overlay/experiments/ExperimentConfig.h')]
if binary.is_file() and all(path.stat().st_mtime_ns <= binary.stat().st_mtime_ns for path in inputs):
    timings['reused'] = True
else:
    timed('client_compile_sec', ['g++', '-O2', '-std=c++17', '-pthread', '-Isrc',
                                 'shab_client.cpp', '-o', 'shab_client'])
print('BUILD_TIMINGS_JSON=' + json.dumps(timings), flush=True)
"""
    # The common client is compiled once; retain its build log under the first protocol.
    build_vm(vm, code, folder, Path(RESULTS_DIR).expanduser().resolve() /
             PROTOCOL / 'builds' / (vm['name'] + '-client-build.log'))


def make_configs(builder, art, trial, node_ips, folder, protocol, scenario):
    code = f'''
from pathlib import Path
import re, shlex, subprocess
repo=Path({REMOTE_BUILD!r}); root=Path({trial!r}); root.mkdir(parents=True)
(root/'ips.txt').write_text({''.join(ip + chr(10) for ip in node_ips)!r})
text=(repo/'gcp_setup_stellar_private.sh').read_text()
text=re.sub(r'^STELLAR_CORE=.*$', 'STELLAR_CORE='+shlex.quote({art + '/stellar-core'!r}), text, flags=re.M)
text=re.sub(r'^BASE_DIR=.*$', 'BASE_DIR='+shlex.quote(str(root/'nodes')), text, flags=re.M)
text=re.sub(r'^mapfile -t NODE_IPS <.*$', 'mapfile -t NODE_IPS < '+shlex.quote(str(root/'ips.txt')), text, flags=re.M)
(root/'setup.sh').write_text(text)
subprocess.run(['bash', str(root/'setup.sh'), 'start'], check=True)
for i in range(1, {NUM_NODES}+1):
    cfg=root/'nodes'/f'node{{i}}'/'stellar-core.cfg'
    text=cfg.read_text()
    options={{'SEND_CUSTOM_MESSAGE': 'true' if i==1 else 'false', 'PUBLIC_HTTP_PORT':'false',
             'TARGET_PEER_CONNECTIONS':str({NUM_NODES}-1), 'MAX_ADDITIONAL_PEER_CONNECTIONS':str(8*{NUM_NODES})}}
    cfg.write_text(''.join(k+'='+v+'\\n' for k,v in options.items())+text)
subprocess.run(['tar','-czf',str(root/'configs.tgz'),'-C',str(root),'nodes'], check=True)
'''
    ssh(builder, "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
        shlex.join(runtime_command(["python3", "-c", code], protocol, scenario)),
        log=folder / "setup.log")
    scp(builder, trial + "/configs.tgz", folder / "configs.tgz", download=True)


def collect_trial_logs(vm, root, local, config):
    filenames = (['stellar-core.log']
                 if vm['role'] == 'node' else ['client.log'])
    diagnostics = []
    # One compressed transfer per VM; omit databases/buckets and retain partial logs on failure.
    code = f'''
import io, json, pathlib, tarfile
root = pathlib.Path({root!r})
root.mkdir(parents=True, exist_ok=True)
missing = []
with tarfile.open(root/'collected-logs.tgz', 'w:gz', compresslevel=1) as archive:
    for name in {filenames!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
        else: missing.append(name)
    for name in {diagnostics!r}:
        if (root/name).is_file(): archive.add(root/name, arcname=name)
    data = json.dumps({{'missing': missing}}).encode()
    member = tarfile.TarInfo('collection.json'); member.size = len(data)
    archive.addfile(member, io.BytesIO(data))
'''
    python_remote(vm, code)
    local.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='stellar-log-transfer-') as transfer_dir:
        archive_path = Path(transfer_dir) / 'collected-logs.tgz'
        scp(vm, root + '/collected-logs.tgz', archive_path, download=True)
        with tarfile.open(archive_path, 'r:gz') as archive:
            missing = json.load(archive.extractfile('collection.json'))['missing']
            for name in filenames + diagnostics:
                if name in archive.getnames():
                    member = archive.getmember(name)
                    if not member.isfile():
                        raise RuntimeError('Expected a regular log file: ' + name)
                    with archive.extractfile(member) as source, (local / name).open('wb') as output:
                        if name == 'client.log':
                            # Keep plot metadata in the log itself, without extra JSON files.
                            output.write(('# RUN_CONFIG_JSON=' + json.dumps(config, allow_nan=False) + '\n').encode())
                        shutil.copyfileobj(source, output)
    if missing:
        raise RuntimeError(vm['name'] + ': missing ' + ', '.join(missing))


def run_trial(vms, builder, art, protocol, scenario, load, repetition, load_index, session, common, client_art=None):
    label = trial_label(protocol, scenario, load, repetition, load_index)
    folder = session / label
    folder.mkdir(parents=True)
    logs_folder = Path(RESULTS_DIR).expanduser().resolve() / label
    logs_folder.mkdir(parents=True, exist_ok=True)
    # A rerun replaces this exact trial. Do not leave old client data behind if
    # the new run fails before collecting logs, and never touch other trials.
    save_json(logs_folder / 'result.json', {'status': 'in_progress'})
    (logs_folder / 'memory.csv').unlink(missing_ok=True)
    for pattern in ('node-*/stellar-core.log', 'client-*/client.log'):
        for old_log in logs_folder.glob(pattern):
            old_log.unlink()
    trial = REMOTE_ROOT + "/runs/" + label
    client_art = client_art or art
    nodes = [v for v in vms if v['role'] == 'node']
    clients = [v for v in vms if v['role'] == 'client'][:load['active_clients']]
    config = {**common, **load, "protocol": protocol, "scenario": scenario, "measured_replica": MEASURED_REPLICA,
              "workload": "WORKLOAD_W", "repetition": repetition, "load_index": load_index,
              "aggregate_max_in_flight": math.prod(load.values()), "remote_trial": trial}
    save_json(folder / "run_config.json", config)
    result = {"status": "failed", "protocol": protocol, "path": label}
    owned = [(v, trial + f"/nodes/node{v['index']}") for v in nodes]
    owned += [(v, trial + f"/client-{v['index']}") for v in clients]
    states = []
    cleanup_errors = []
    timings = {}
    setup_start = time.monotonic()
    print(f'Starting {label}: setup, then {CLIENT_START_LEAD_SEC}s scheduled client lead.', flush=True)
    try:
        def reset_trial(vm):
            stop_vm_processes(vm)
            ssh(vm, shlex.join(["rm", "-rf", "--", trial]))
        parallel(reset_trial, vms)
        make_configs(builder, art, trial, [v['ip'] for v in nodes], folder, protocol, scenario)
        def prepare_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            ssh(vm, shlex.join(["mkdir", "-p", trial]))
            scp(vm, folder / "configs.tgz", trial + "/configs.tgz")
            ssh(vm, shlex.join(["tar", "-xzf", trial + "/configs.tgz", "-C", trial]) + "\n" +
                "export LD_LIBRARY_PATH=" + shlex.quote(art + "/lib") + "\n" +
                shlex.join(runtime_command([art + "/stellar-core", "new-db", "--conf",
                                            root + "/stellar-core.cfg"], protocol, scenario)),
                log=folder / f"node-{vm['index']}-init.log")
        parallel(prepare_node, nodes)
        def start_node(vm):
            root = trial + f"/nodes/node{vm['index']}"
            launch_process(vm, root, runtime_command(
                [art + "/stellar-core", "run", "--conf", root + "/stellar-core.cfg"], protocol, scenario),
                           art + "/lib",
                           "console.log")
        parallel(start_node, nodes)
        deadline = time.monotonic() + STARTUP_TIMEOUT_SEC
        last_connectivity = None
        while True:
            status = parallel(lambda v: process_state(v, trial + f"/nodes/node{v['index']}", check_ready=True), nodes)
            if any(s['done'] or s.get('error') for s in status):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise RuntimeError('Replica exited during startup: ' + str(result['replica_outcomes']))
            connectivity = (sum(s['ready'] for s in status),
                            min(s.get('connected_peers', 0) for s in status))
            if connectivity != last_connectivity:
                print(f"Connectivity: {connectivity[0]}/{len(nodes)} replicas fully connected; "
                      f"minimum {connectivity[1]}/{len(nodes)-1} authenticated peers", flush=True)
                last_connectivity = connectivity
            if all(s['ready'] for s in status) and status[0]['listener']:
                break
            if time.monotonic() >= deadline:
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], status))
                raise TimeoutError("Full peer connectivity/client listener timed out: " +
                                   str(result['replica_outcomes']))
            time.sleep(3)
        start_at = time.time() + CLIENT_START_LEAD_SEC if CLIENT_START_LEAD_SEC else 0
        timings['setup_sec'] = round(time.monotonic() - setup_start, 3)
        config['scheduled_client_start'] = start_at or None
        save_json(folder / "run_config.json", config)
        def start_client(vm):
            launch_process(vm, trial + f"/client-{vm['index']}",
                runtime_command([client_art + "/shab_client", nodes[0]['ip'], str(CLIENT_PORT), str(load['max_in_flight']),
                 "2000000000", str(SERVER_BATCH_SIZE), str(CLIENT_DURATION_SEC),
                 str(SEND_INTERVAL_US), str(load['client_threads'])], protocol, scenario),
                client_art + "/lib", "client.log", start_at=start_at)
        parallel(start_client, clients)
        if start_at and time.time() >= start_at:
            raise RuntimeError("Client scheduling exceeded lead time; increase CLIENT_START_LEAD_SEC")
        workload_end = (start_at or time.time()) + CLIENT_DURATION_SEC
        while time.time() < workload_end:
            # Poll replicas and clients in one batch; neither group depends on the other.
            observed = parallel(lambda pair: process_state(*pair), owned)
            replica_states, states = observed[:len(nodes)], observed[len(nodes):]
            if any(s['done'] or s.get('error') for s in replica_states):
                result['replica_outcomes'] = dict(zip([v['name'] for v in nodes], replica_states))
                raise RuntimeError('Replica exited during workload: ' + str(result['replica_outcomes']))
            if any(s.get('error') for s in states):
                raise RuntimeError('Client process failed: ' + str(states))
            if all(s['done'] for s in states):
                break
            time.sleep(min(3, max(0, workload_end - time.time())))
        result['client_outcomes'] = states
        if any('started_at' not in s for s in states):
            raise RuntimeError('A client never started: ' + str(states))
        starts = [s['started_at'] for s in states]
        result['client_started_at'] = min(starts)
        result['client_start_skew_sec'] = max(starts) - min(starts)
        if result['client_start_skew_sec'] > MAX_CLIENT_START_SKEW_SEC:
            raise RuntimeError("Client start skew exceeded configured limit")
        if any(s.get('error') or (s['done'] and s['returncode'] != 0) for s in states):
            raise RuntimeError("A client failed; inspect client logs")
        result['status'] = 'ok'
        result['stop_reason'] = 'workload_duration_reached' if time.time() >= workload_end else 'clients_completed'
    except Exception as exc:
        result.update(status='failed', error=str(exc))
    finally:
        # On Ctrl-C this block still stops owned processes and tries to collect logs.
        cleanup_errors = []
        stop_start = time.monotonic()
        def stop(pair):
            vm, root = pair
            try:
                stop_vm_processes(vm)
            except Exception as exc:
                cleanup_errors.append(f"{vm['name']}: {exc}")
        parallel(stop, owned)
        timings['stop_sec'] = round(time.monotonic() - stop_start, 3)
        collect_start = time.monotonic()
        def collect(pair):
            vm, root = pair
            local = logs_folder / (f"node-{vm['index']}" if vm['role']=='node' else f"client-{vm['index']}")
            collect_trial_logs(vm, root, local, config)
        try:
            parallel(collect, owned)
        except Exception as exc:
            cleanup_errors.append('Log download: ' + str(exc))
        timings['collect_sec'] = round(time.monotonic() - collect_start, 3)
        if cleanup_errors:
            result.update(status='failed', cleanup_errors=cleanup_errors)
        result['config'] = config
        if result['status'] == 'ok':
            try:
                result["sample_count"] = save_memory_samples(logs_folder, config, result["client_started_at"])
            except (OSError, ValueError) as exc:
                result.update(status='failed', error=str(exc))
        result['timings_sec'] = timings
        save_json(logs_folder / "result.json", result)
        print(f"Cleanup {label}: stop={timings['stop_sec']:.1f}s, logs={timings['collect_sec']:.1f}s", flush=True)
    print(label, result['status'], result.get('error', ''), flush=True)
    if cleanup_errors:
        raise RuntimeError(result.get('error', 'Trial cleanup failed') + '\nCleanup: ' +
                           '; '.join(cleanup_errors) + '\nLogs: ' + str(logs_folder))
    return result


def save_memory_samples(folder, cfg, client_start):
    # VmRSS and prepareHistorySize() come from the same once-per-second replica tick.
    pattern = re.compile(r"MEMORY RSS: (\d+) MB, number of elements: (\d+), "
                         r"view=(\d+) unix_us=(\d+) nonprune=(true|false)")
    expected = "true" if cfg["scenario"] == "never_pruned" else "false"
    rows = []
    with (folder / f"node-{cfg['measured_replica']}" / "stellar-core.log").open(errors="replace") as log:
        for line in log:
            m = pattern.search(line)
            if not m:
                continue
            rss, ballots, view, stamp = map(int, m.groups()[:4])
            if m[5] != expected:
                raise ValueError("Replica logged a different pruning mode than requested")
            elapsed = stamp / 1e6 - client_start
            if 0 <= elapsed < cfg["client_duration_sec"]:
                if rss <= 0:
                    raise ValueError("Replica reported no Linux VmRSS measurement")
                rows.append((elapsed, view, ballots, rss))
    if len(rows) < 2 or rows[-1][1] <= rows[0][1]:
        raise ValueError("Need view progress and at least two memory samples during the client workload")
    with (folder / "memory.csv").open("w", newline="") as out:
        writer = csv.writer(out)
        writer.writerow(("client_elapsed_sec", "view", "retained_ballots", "rss_mib"))
        writer.writerows(rows)
    return len(rows)


def trial_label(protocol, scenario, load, repetition, index):
    return (f"{protocol}/n{NUM_NODES}/{scenario}/node{MEASURED_REPLICA}/repeat{repetition}/load{index}_c{load['active_clients']}"
            f"_t{load['client_threads']}_w{load['max_in_flight']}")


def stop_vm_processes(vm, *, include_builds=False):
    # These are dedicated experiment VMs. Restrict cleanup to the SSH user's processes.
    # Bracketed dots keep pkill from matching the cleanup command itself.
    pattern = re.escape(REMOTE_ROOT) + r'/[^ ]*/(launch[.]sh|worker[.]py supervise )'
    script = """
kill_matching() {
    pkill -u "$(id -u)" "$@" || { rc=$?; [ "$rc" -eq 1 ]; }
}
kill_matching -TERM -f PATTERN
kill_matching -TERM -x stellar-core
kill_matching -TERM -x shab_client
if pgrep -x stellar-core >/dev/null || pgrep -x shab_client >/dev/null; then
    sleep 1
fi
kill_matching -KILL -x stellar-core
kill_matching -KILL -x shab_client
kill_matching -KILL -f PATTERN
""".replace('PATTERN', shlex.quote(pattern))
    if include_builds:
        # Kill old notebook build supervisors first, so they cannot start another
        # make/compiler after cleanup. The bracket prevents matching this script.
        # Dedicated VMs may have orphaned children after a kernel/SSH interruption.
        build_cleanup = """
kill_matching -KILL -f '[B]UILD_TIMINGS_JSON='
kill_matching -KILL -x '(make|gmake|ninja|ninja-build|cmake|cargo)(-[0-9.]+)?'
kill_matching -KILL -x '(gcc|g[+][+]|cc|c[+][+]|clang|clang[+][+]|cc1|cc1plus|collect2|lto-wrapper|lto1|as|ld|ld[.]bfd|ld[.]gold|ld[.]lld|lld|rustc|rustdoc|build-script-bu)(-[0-9.]+)?'
"""
        # Insert after the shell helper, before stopping replicas and clients.
        script = script.replace("\n}\n", "\n}\n" + build_cleanup, 1)
    ssh(vm, script, timeout=30)


def prepare_build(vm, sha, folder):
    # Update the existing checkout once. Never copy it, clean it, or reinstall dependencies.
    script = f"""
cd {shlex.quote(IMAGE_REPO)}
test -d .git
test -f Makefile
if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
    export GIT_TERMINAL_PROMPT=0
    # The sweep itself edits this header; clear those generated flags before pulling.
    if ! git diff --quiet HEAD -- src/overlay/experiments/ExperimentConfig.h; then
        git restore --source=HEAD -- src/overlay/experiments/ExperimentConfig.h
    fi
    git pull --ff-only origin {shlex.quote(sha)}
    if [ "$(git rev-parse HEAD)" != {shlex.quote(sha)} ]; then
        echo "Existing checkout is ahead of the session commit; refusing to mix source versions." >&2
        exit 1
    fi
fi
printf 'Using existing configured checkout: %s\\n' "$PWD"
"""
    ssh(vm, script, log=folder / 'prepare-build.log', timeout=600)


def finish_session(session):
    print('Finished:', Path(RESULTS_DIR).expanduser().resolve(),
          '\nRun the GCP memory plotting cell in PostProcess.ipynb with the same RESULTS_DIR.')
    saved = json.loads((session / 'fleet.json').read_text()) if (session / 'fleet.json').exists() else {'vms': []}
    if DELETE_ALL_AFTER_FINAL_RUN:
        # Resume deletion too, without recreating VMs already deleted by a prior attempt.
        existing = {(v['name'], v['zone'].split('/')[-1]) for v in
                    json.loads(cloud('instances', 'list', '--format=json'))}
        remaining = [v for v in saved['vms'] if (v['name'], v['zone']) in existing]
        print('Deleting remaining replica/client VMs:', ', '.join(v['name'] for v in remaining), flush=True)
        parallel(lambda vm: cloud('instances', 'delete', vm['name'], '--zone', vm['zone'], timeout=900),
                 remaining)
    else:
        print('VMs kept running for reuse (DELETE_ALL_AFTER_FINAL_RUN=False).')


def main_gcp_memory(work_dir):
    global REMOTE_BUILD
    repo = REPO.expanduser().resolve()
    if PROTOCOL != "shabdiz":
        raise ValueError("Memory trials require Shabdiz")
    if type(RESUME_SESSION) is not bool:
        raise ValueError('RESUME_SESSION must be True or False')
    if not re.fullmatch(r'[a-z][a-z0-9-]{0,39}', INSTANCE_PREFIX):
        raise ValueError('INSTANCE_PREFIX must be a short lowercase GCP name')
    if any(not re.fullmatch(r'/[A-Za-z0-9_./-]+', p) for p in (REMOTE_ROOT, IMAGE_REPO)):
        raise ValueError('Remote paths must be absolute, without spaces or shell metacharacters')
    if not 4 <= NUM_NODES <= 256 or len(REPLICA_ZONES) != NUM_NODES:
        raise ValueError("Specify one zone per replica and 4..256 replicas")
    if not SCENARIOS or len(set(SCENARIOS)) != len(SCENARIOS) or set(SCENARIOS) - {"never_pruned", "normal_pruning"}:
        raise ValueError("Choose distinct pruning scenarios")
    if not 1 <= MEASURED_REPLICA <= NUM_NODES:
        raise ValueError("MEASURED_REPLICA is a one-based replica index")
    if type(MAX_PARALLEL_WORKERS) is not int or MAX_PARALLEL_WORKERS < 1:
        raise ValueError('MAX_PARALLEL_WORKERS must be a positive integer')
    if min(CLIENT_DURATION_SEC, REPETITIONS, BUILD_JOBS, SERVER_BATCH_SIZE) <= 0:
        raise ValueError("Duration, counts and batch size must be positive")
    if not LOAD_POINTS:
        raise ValueError("No client load points")
    for load in LOAD_POINTS:
        if set(load) != {'active_clients','client_threads','max_in_flight'} or min(load.values()) <= 0:
            raise ValueError(f"Invalid load point: {load}")
        if load['active_clients'] > len(CLIENT_ZONES) or math.prod(load.values()) < SERVER_BATCH_SIZE:
            raise ValueError("Insufficient client VMs or aggregate in-flight capacity for one batch")
    sha = header = None
    if not RESUME_SESSION:
        sha = command(['git', '-C', repo, 'rev-parse', (SOURCE_COMMIT or 'HEAD') + '^{commit}']).strip()
        header = command(['git', '-C', repo, 'show', sha + ':src/overlay/experiments/ExperimentConfig.h'])
        require_runtime_config(header)
    session_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
    # Session bookkeeping is temporary; raw logs use fixed per-trial paths.
    session = Path(work_dir)
    REMOTE_BUILD = IMAGE_REPO
    common = {'schema_version': 1, 'session': session_id, 'source_commit': sha, 'num_nodes': NUM_NODES,
              'batch_size': SERVER_BATCH_SIZE, 'client_duration_sec': CLIENT_DURATION_SEC,
              'send_interval_us': SEND_INTERVAL_US, 'image_family': IMAGE_FAMILY,
              'machine_type': MACHINE_TYPE, 'client_machine_type': CLIENT_MACHINE_TYPE,
              'project': PROJECT, 'image_project': IMAGE_PROJECT, 'remote_root': REMOTE_ROOT,
              'image_repo': IMAGE_REPO, 'replica_zones': REPLICA_ZONES, 'client_zones': CLIENT_ZONES,
              'client_port': CLIENT_PORT, 'client_drain_timeout_sec': 0}
    manifest = {**common, 'protocol': PROTOCOL, 'scenarios': SCENARIOS, 'load_points': LOAD_POINTS,
                'repetitions': REPETITIONS, 'instance_prefix': INSTANCE_PREFIX,
                'delete_all_after_final_run': DELETE_ALL_AFTER_FINAL_RUN, 'resume_session': RESUME_SESSION}
    session.mkdir(parents=True, exist_ok=True)
    save_json(session / 'session.json', manifest)
    print('Results:', Path(RESULTS_DIR).expanduser().resolve(), flush=True)
    trials = [(scenario, repetition, index, load) for scenario in SCENARIOS
              for repetition in range(1, REPETITIONS+1) for index, load in enumerate(LOAD_POINTS, 1)]
    results = []
    save_json(session / 'results.json', results)
    print(f"{len(trials)} trials; "
          f"{len(trials)*CLIENT_DURATION_SEC/3600:.1f} "
          'hours of workload, plus builds and setup; pkill at workload end.', flush=True)
    vms = fleet(session)
    parallel(wait_ssh, vms)
    common['replica_zones'] = REPLICA_ZONES
    common['client_zones'] = CLIENT_ZONES
    common['resolved_image'] = json.loads((session/'fleet.json').read_text())['image']
    try:
        print('Stopping stale replicas, clients, and builds on all experiment VMs...', flush=True)
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
        def initialize_vm(vm):
            sync = ssh(vm, shlex.join(['mkdir', '-p', REMOTE_ROOT]) + '\n' +
                       'timedatectl show --property=NTPSynchronized --value').strip()
            if sync != 'yes':
                raise RuntimeError(f"{vm['name']}: synchronize VM clock before scheduling clients")
        parallel(initialize_vm, vms)
        builder = next(v for v in vms if v['role']=='node')
        client_art = IMAGE_REPO
        def prepare_vm(vm):
            folder = session / 'preparation' / vm['name']
            folder.mkdir(parents=True, exist_ok=True)
            prepare_build(vm, sha, folder)
        if RESUME_SESSION:
            print('Reusing VM source; checking common replica/client builds once.', flush=True)
            def existing_source(vm):
                check = 'test -f shab_client.cpp' if vm['role'] == 'client' else 'test -f Makefile'
                return ssh(vm, 'cd ' + shlex.quote(IMAGE_REPO) + '\n' + check + '\ngit rev-parse HEAD').strip()
            sources = parallel(existing_source, vms)
            if len(set(sources)) != 1:
                raise RuntimeError('VM source revisions differ. Run once with RESUME_SESSION=False to update them.')
            sha = sources[0]
            header = ssh(builder, 'cd ' + shlex.quote(IMAGE_REPO) +
                         '\ngit show HEAD:src/overlay/experiments/ExperimentConfig.h')
            require_runtime_config(header)
        else:
            print(f'Updating existing {IMAGE_REPO} once before the common build.', flush=True)
            parallel(prepare_vm, vms)
        common['source_commit'] = manifest['source_commit'] = sha
        save_json(session / 'session.json', manifest)
        print('Source:', sha, flush=True)
        # A single binary handles all protocols and node counts. make is incremental;
        # the client timestamp check also runs in resume mode to avoid stale binaries.
        print(f'Checking common builds once ({BUILD_JOBS} make jobs per replica)...', flush=True)
        build_folder = session / 'builds' / PROTOCOL
        art = IMAGE_REPO + '/src'
        build_start = time.monotonic()
        def build_common(vm):
            folder = build_folder / vm['name']
            if vm['role'] == 'node':
                build_replica(vm, header, folder)
            else:
                build_client(vm, header, folder)
        parallel(build_common, vms)
        print(f'Common binaries ready in {time.monotonic() - build_start:.1f}s', flush=True)
        for scenario, repetition, index, load in trials:
            result = run_trial(vms, builder, art, PROTOCOL, scenario, load, repetition, index,
                               session, common, client_art)
            results.append(result)
            save_json(session / 'results.json', results)
    finally:
        parallel(lambda vm: stop_vm_processes(vm, include_builds=True), vms)
    failed = [result for result in results if result['status'] not in ('ok', 'drain_timeout')]
    if failed:
        print(f'{len(failed)} trials failed; VMs kept. Set RESUME_SESSION=True to rerun all trials with existing builds.')
    else:
        finish_session(session)
    return Path(RESULTS_DIR).expanduser().resolve()


def run_gcp_memory():
    global RESULTS_DIR
    RESULTS_DIR = RESULTS_ROOT.expanduser().resolve()
    Path(RESULTS_DIR).expanduser().resolve().mkdir(parents=True, exist_ok=True)
    # Config archives, setup logs, timing files, and status JSON are temporary.
    # Per-trial status and logs remain available for plotting.
    with tempfile.TemporaryDirectory(prefix='stellar-gcp-shabdiz-memory-') as work_dir:
        return main_gcp_memory(work_dir)


GCP_MEMORY_RESULTS = run_gcp_memory()

# SCP

In [ ]:
# import subprocess
# import concurrent.futures
# import shutil
# import time
# import re
# from pathlib import Path
# from datetime import datetime


# # =============================================================================
# # Basic helpers
# # =============================================================================

# def run_shell(command, check=False):
#     print(f"\n$ {command}")
#     rc = subprocess.call(command, shell=True)
#     if check and rc != 0:
#         raise RuntimeError(f"Command failed with code {rc}: {command}")
#     return rc


# def run_shell_output(command):
#     print(f"\n$ {command}")
#     return subprocess.check_output(command, shell=True).decode()


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Experiment config
# # =============================================================================

# PROJECT = "research-488322"
# ZONE = "us-central1-b"
# MACHINE_TYPE = "e2-standard-2"
# IMAGE_FAMILY = "tsm-sc-family"
# SUBNET = "default"
# GCP_USERNAME = "tejas"

# # Run in decreasing order so we can delete unused nodes after each subrun.
# NUM_NODES_LIST = [48, 32, 16, 8, 4]
# MAX_NUM_NODES = max(NUM_NODES_LIST)

# N_CLIENTS = 0

# # You said NUM_TEST_ACCOUNTS is already set to 100 in C++.
# NUM_TEST_ACCOUNTS = 100

# # This is only used for summary labels. It is not patched by this script.
# SCP_OPS_PER_TX = 100

# RUN_DURATION_SEC = 210
# MEASURE_START_SEC = 120
# MEASURE_END_SEC = 180

# LOCAL_PRIVATE_DIR = Path("../stellar-private")
# RESULTS_DIR = Path("/home/tejas/work/experiments/stellar_native")
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# DELETE_BEFORE_FIRST_RUN = False

# # Final cleanup deletes whatever remains after the 4-node run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile only during the first subrun.
# COMPILE_ONLY_FIRST_SUBRUN = True


# # =============================================================================
# # GCP helpers
# # =============================================================================

# def instance_name(i):
#     return f"tsm-sc-{i:03}"


# def get_zone_for_instance(i):
#     return ZONE


# def fetch_existing_instances():
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance
#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def delete_instance_by_index(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting unused node {name}")
#     return subprocess.call(cmd, shell=True)


# def delete_nodes_not_needed_next(current_num_nodes, next_num_nodes):
#     """
#     Because NUM_NODES_LIST is descending:
#       after 48-node run, delete 32..47
#       after 32-node run, delete 16..31
#       after 16-node run, delete 8..15
#       after 8-node run, delete 4..7
#     """
#     if next_num_nodes is None:
#         return

#     to_delete = list(range(next_num_nodes, current_num_nodes))

#     if not to_delete:
#         return

#     print(
#         f"Deleting nodes no longer needed for future subruns: "
#         f"{instance_name(to_delete[0])} to {instance_name(to_delete[-1])}"
#     )

#     run_parallel(
#         delete_instance_by_index,
#         to_delete,
#         max_workers=min(32, len(to_delete)),
#     )


# def create_instance(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {name} \
#         --project={PROJECT} \
#         --zone={inst_zone} \
#         --machine-type={MACHINE_TYPE} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={SUBNET} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={IMAGE_FAMILY},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     print(f"Creating {name}")
#     return subprocess.call(cmd, shell=True)


# def ensure_instances():
#     existing = fetch_existing_instances()
#     existing_names = {name for name, _ in existing}

#     needed_indices = list(range(MAX_NUM_NODES + N_CLIENTS))
#     missing = [i for i in needed_indices if instance_name(i) not in existing_names]

#     if not missing:
#         print("All required instances already exist.")
#         return

#     print("Missing instances:", [instance_name(i) for i in missing])
#     run_parallel(create_instance, missing, max_workers=min(32, len(missing)))

#     print("Waiting after VM creation...")
#     time.sleep(30)


# def get_node_records(num_nodes):
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()

#     records = []
#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             records.append((idx, name, inst_zone, ip))

#     records.sort()
#     node_records = [r for r in records if r[0] < num_nodes]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} nodes, found {len(node_records)}: {node_records}"
#         )

#     return node_records


# def kill_stellar(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo pkill -9 stellar-core || true; \
# sudo pkill -9 shab_client || true; \
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def git_pull_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# git pull
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def compile_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# make -j2; \
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def clean_remote_private(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def copy_private_to_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# --recurse "/home/tejas/stellar-private" "{name}:/home/tejas/stellar-private"
# """
#     return subprocess.call(cmd, shell=True)


# def run_stellar_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def collect_node_log(i, local_run_dir):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     local_run_dir.mkdir(parents=True, exist_ok=True)
#     local_path = local_run_dir / f"{name}-node{node_number}-stellar-core.log"

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# "{name}:/home/tejas/stellar-private/node{node_number}/stellar-core.log" \
# "{local_path}"
# """
#     return subprocess.call(cmd, shell=True)


# # =============================================================================
# # Config generation
# # =============================================================================

# def write_tsm_ips(node_ips):
#     with open("tsm_ips.txt", "w") as f:
#         for ip in node_ips:
#             f.write(ip + "\n")


# def generate_stellar_private_configs(num_nodes):
#     if LOCAL_PRIVATE_DIR.exists():
#         shutil.rmtree(LOCAL_PRIVATE_DIR)

#     LOCAL_PRIVATE_DIR.mkdir(parents=True)

#     run_shell(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     run_shell(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     # Only node1 injects native Stellar workload.
#     # ENABLE_SCP_TRACKING is already true in C++.
#     for node_number in range(1, num_nodes + 1):
#         cfg = LOCAL_PRIVATE_DIR / f"node{node_number}" / "stellar-core.cfg"

#         if not cfg.exists():
#             raise FileNotFoundError(f"Missing config: {cfg}")

#         lines = cfg.read_text().splitlines()
#         lines = [
#             line for line in lines
#             if not line.strip().startswith("SEND_CUSTOM_MESSAGE=")
#         ]

#         if node_number == 1:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=true")
#         else:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=false")

#         cfg.write_text("\n".join(lines) + "\n")


# # =============================================================================
# # Log parsing
# # =============================================================================

# def parse_ts(line):
#     # Example:
#     # 2026-06-30T00:09:11.842 GDQOM [Ledger INFO] Got consensus...
#     m = re.match(r"^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?)", line)
#     if not m:
#         return None

#     try:
#         return datetime.fromisoformat(m.group(1))
#     except ValueError:
#         return None


# def parse_committed_ops(log_path, start_sec=MEASURE_START_SEC, end_sec=MEASURE_END_SEC):
#     entries = []

#     with open(log_path, "r", errors="ignore") as f:
#         for line in f:
#             if "Got consensus:" not in line:
#                 continue

#             ts = parse_ts(line)
#             if ts is None:
#                 continue

#             m_ops = re.search(r"ops=(\d+)", line)
#             m_txs = re.search(r"txs=(\d+)", line)

#             if not m_ops:
#                 continue

#             ops = int(m_ops.group(1))
#             txs = int(m_txs.group(1)) if m_txs else 0

#             entries.append((ts, txs, ops, line.rstrip()))

#     if not entries:
#         return {
#             "ops": 0,
#             "txs": 0,
#             "throughput_ops_s": 0.0,
#             "avg_ops_per_ledger": 0.0,
#             "avg_txs_per_ledger": 0.0,
#             "min_ops_per_ledger": 0,
#             "max_ops_per_ledger": 0,
#             "min_txs_per_ledger": 0,
#             "max_txs_per_ledger": 0,
#             "num_ledgers": 0,
#             "first_ts": None,
#             "last_ts": None,
#         }

#     t0 = entries[0][0]
#     selected = []

#     for ts, txs, ops, line in entries:
#         elapsed = (ts - t0).total_seconds()
#         if start_sec <= elapsed <= end_sec:
#             selected.append((ts, txs, ops, line))

#     total_ops = sum(ops for _, _, ops, _ in selected)
#     total_txs = sum(txs for _, txs, _, _ in selected)
#     num_ledgers = len(selected)
#     window = max(1.0, end_sec - start_sec)

#     ops_values = [ops for _, _, ops, _ in selected]
#     txs_values = [txs for _, txs, _, _ in selected]

#     return {
#         "ops": total_ops,
#         "txs": total_txs,
#         "throughput_ops_s": total_ops / window,
#         "avg_ops_per_ledger": total_ops / num_ledgers if num_ledgers else 0.0,
#         "avg_txs_per_ledger": total_txs / num_ledgers if num_ledgers else 0.0,
#         "min_ops_per_ledger": min(ops_values) if ops_values else 0,
#         "max_ops_per_ledger": max(ops_values) if ops_values else 0,
#         "min_txs_per_ledger": min(txs_values) if txs_values else 0,
#         "max_txs_per_ledger": max(txs_values) if txs_values else 0,
#         "num_ledgers": num_ledgers,
#         "first_ts": selected[0][0].isoformat() if selected else None,
#         "last_ts": selected[-1][0].isoformat() if selected else None,
#     }


# def write_summary(run_dir, num_nodes, leader_log):
#     summary = parse_committed_ops(leader_log)

#     summary_path = run_dir / "summary.txt"

#     with open(summary_path, "w") as f:
#         f.write(f"num_nodes={num_nodes}\n")
#         f.write(f"num_accounts={NUM_TEST_ACCOUNTS}\n")
#         f.write(f"ops_per_tx={SCP_OPS_PER_TX}\n")
#         f.write(f"measurement_window={MEASURE_START_SEC}-{MEASURE_END_SEC}s\n")
#         f.write(f"committed_ops={summary['ops']}\n")
#         f.write(f"committed_txs={summary['txs']}\n")
#         f.write(f"throughput_ops_s={summary['throughput_ops_s']:.2f}\n")
#         f.write(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}\n")
#         f.write(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}\n")
#         f.write(f"min_ops_per_ledger={summary['min_ops_per_ledger']}\n")
#         f.write(f"max_ops_per_ledger={summary['max_ops_per_ledger']}\n")
#         f.write(f"min_txs_per_ledger={summary['min_txs_per_ledger']}\n")
#         f.write(f"max_txs_per_ledger={summary['max_txs_per_ledger']}\n")
#         f.write(f"num_ledgers={summary['num_ledgers']}\n")
#         f.write(f"first_ts={summary['first_ts']}\n")
#         f.write(f"last_ts={summary['last_ts']}\n")

#     print("\n" + "=" * 80)
#     print(f"RESULT num_nodes={num_nodes}")
#     print(f"committed_ops={summary['ops']}")
#     print(f"committed_txs={summary['txs']}")
#     print(f"throughput_ops_s={summary['throughput_ops_s']:.2f}")
#     print(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}")
#     print(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}")
#     print(f"min_ops_per_ledger={summary['min_ops_per_ledger']}")
#     print(f"max_ops_per_ledger={summary['max_ops_per_ledger']}")
#     print(f"num_ledgers={summary['num_ledgers']}")
#     print(f"summary={summary_path}")
#     print("=" * 80 + "\n")

#     return summary


# import os
# os.system('git add .; git commit -m "update"; git push')

# # =============================================================================
# # Main experiment
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()
#     if instances:
#         run_parallel(delete_instance, instances, max_workers=min(32, len(instances)))

# ensure_instances()

# max_node_records = get_node_records(MAX_NUM_NODES)

# print("All available node records:")
# for r in max_node_records:
#     print(r)

# all_results = []

# os.system('git add .; git commit -m "update"; git push')

# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
#     run_name = f"stellar_native_nodes_{num_nodes}_accounts_{NUM_TEST_ACCOUNTS}"
#     run_dir = RESULTS_DIR / run_name
#     run_dir.mkdir(parents=True, exist_ok=True)

#     print("\n" + "#" * 100)
#     print(
#         f"Starting Stellar native run: "
#         f"nodes={num_nodes}, accounts={NUM_TEST_ACCOUNTS}, opsPerTx={SCP_OPS_PER_TX}"
#     )
#     print("#" * 100)

#     active_indices = list(range(num_nodes))
#     active_node_records = get_node_records(num_nodes)
#     active_node_ips = [r[3] for r in active_node_records]

#     print("Active node records:")
#     for r in active_node_records:
#         print(r)

#     print("Active node IPs:", active_node_ips)

#     # Kill all currently-existing nodes before each subrun.
#     # For first run this is 48; after deletion it becomes smaller.
#     existing_before_run = fetch_existing_instances()
#     existing_indices = []
#     for name, _ in existing_before_run:
#         if name.startswith("tsm-sc-"):
#             try:
#                 idx = int(name.rsplit("-", 1)[1])
#                 existing_indices.append(idx)
#             except ValueError:
#                 pass

#     if existing_indices:
#         print("Killing Stellar on all existing tsm-sc nodes before subrun...")
#         run_parallel(
#             kill_stellar,
#             sorted(existing_indices),
#             max_workers=min(48, len(existing_indices)),
#         )

#     # Compile only once, during the first subrun.
#     if (not COMPILE_ONLY_FIRST_SUBRUN) or (run_idx == 0):
#         print("Compiling Stellar Core on all max nodes. This happens only for first subrun.")

#         run_parallel(
#             git_pull_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )

#         run_parallel(
#             compile_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )
#     else:
#         print("Skipping compile for this subrun.")


#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Generate configs for the current node count.
#     # This rewrites tsm_ips.txt so the generated private network has exactly num_nodes validators.
#     write_tsm_ips(active_node_ips)
#     generate_stellar_private_configs(num_nodes)

#     # Clean and copy configs only to active nodes.
#     run_parallel(
#         clean_remote_private,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     run_parallel(
#         copy_private_to_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Start only active nodes.
#     run_parallel(
#         run_stellar_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print(f"Nodes started. Running for {RUN_DURATION_SEC} seconds...")
#     time.sleep(RUN_DURATION_SEC)

#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print("Collecting logs...")
#     run_parallel(
#         lambda i: collect_node_log(i, run_dir),
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     leader_log = run_dir / "tsm-sc-000-node1-stellar-core.log"

#     if leader_log.exists():
#         result = write_summary(run_dir, num_nodes, leader_log)
#         result["num_nodes"] = num_nodes
#         all_results.append(result)
#     else:
#         print(f"WARNING: leader log not found: {leader_log}")

#     # Delete nodes that are no longer needed for the next smaller subrun.
#     if run_idx + 1 < len(NUM_NODES_LIST):
#         next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#         delete_nodes_not_needed_next(num_nodes, next_num_nodes)


# # =============================================================================
# # Final CSV summary
# # =============================================================================

# csv_path = RESULTS_DIR / f"stellar_native_accounts_{NUM_TEST_ACCOUNTS}_nodes_sweep_summary.csv"

# with open(csv_path, "w") as f:
#     f.write(
#         "num_nodes,num_accounts,committed_ops,committed_txs,throughput_ops_s,"
#         "avg_ops_per_ledger,avg_txs_per_ledger,"
#         "min_ops_per_ledger,max_ops_per_ledger,"
#         "min_txs_per_ledger,max_txs_per_ledger,"
#         "num_ledgers,first_ts,last_ts\n"
#     )

#     for r in all_results:
#         f.write(
#             f"{r['num_nodes']},"
#             f"{NUM_TEST_ACCOUNTS},"
#             f"{r['ops']},"
#             f"{r['txs']},"
#             f"{r['throughput_ops_s']:.2f},"
#             f"{r['avg_ops_per_ledger']:.2f},"
#             f"{r['avg_txs_per_ledger']:.2f},"
#             f"{r['min_ops_per_ledger']},"
#             f"{r['max_ops_per_ledger']},"
#             f"{r['min_txs_per_ledger']},"
#             f"{r['max_txs_per_ledger']},"
#             f"{r['num_ledgers']},"
#             f"{r['first_ts']},"
#             f"{r['last_ts']}\n"
#         )

# print(f"\nFinal CSV summary: {csv_path}")

# if DELETE_ALL_AFTER_FINAL_RUN:
#     print("Deleting remaining tsm-sc instances after final run...")
#     instances = fetch_existing_instances()
#     tsm_instances = [
#         (name, zone)
#         for name, zone in instances
#         if name.startswith("tsm-sc-")
#     ]

#     run_parallel(
#         delete_instance,
#         tsm_instances,
#         max_workers=min(32, len(tsm_instances)),
#     )

In [ ]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")